# MERGED Consolidated PDP + CFD + UFD Wedge Engine

This notebook preserves the original downstream QA, plotting, rankings, RTP sensitivity, and current-production diagnostics from the uploaded notebook. The upstream CFD section is updated to use the CFD Master and the 01-05 exports apply the Hercules oil/gas rule.

# Consolidated PDP + Confirmed + Uncertain Development Wedge Engine

CSV-only notebook for JupyterLite.

**Inputs**
- `massive_PDP_production.csv`
- `massive_PDP_frcst.csv`
- `PDProutingexample.csv`: A Facility, B Well Name, E WI %
- `CFDroutingexample.csv`: A Facility, B Well Name, D WI %
- `Confirmed_Dev_Well_Production.csv`: horizontal repeating well blocks
- `UncertainDevRoutingExample.csv`: A Facility, B Well Name, C WI %, D TC Connection, E POP Date
- `MONTHLY_FORECAST_6COL.csv`: A TC/ENTITY_NAME, B Forecast Month, C Oil BBL/month, E Gas MCF/month

**Rules**
- Routing files are hard whitelists: source-file wells not routed are ignored.
- WI is converted from percent to decimal and applied at well-month level before aggregation.
- All PDP, confirmed-dev, and uncertain-dev wells are capped at **500 months**.
- PDP uses actual production through each well's last production month, then forecast.
- Confirmed dev starts its 500-month window at the first nonzero oil/gas month.
- Uncertain dev POP is rounded to the nearest first of month; Forecast Month 1 is assigned to that full month.

**Cases**
1. PDP Base
2. PDP + Confirmed Development
3. PDP + Confirmed + Uncertain Development

Each facility gets oil and gas rate+cumulative plots with peak rate, rate at 2/1/2034, and end cumulative annotations. Development cases show the incremental rate wedge. Exactly six monthly CSV outputs are written, one per case/fluid.


In [ ]:
import os
print("Current dir:", os.getcwd())
print("Files are:", os.listdir())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, os, gc

# ============================================================
# INPUT FILES
# ============================================================
PDP_PRODUCTION_FILE = "MASSIVE_PDP_PRD_0902_715.csv"
PDP_FORECAST_FILE   = "MASSIVE_PDP_FRCST_0902_715.csv"
PDP_ROUTING_FILE    = "PDProutingexample_0902.csv"

# NEW CFD MASTER (A:G = Facility, Well, WI %, Production Method,
# POP Date, Forecast CSV Location, Production Link)
CFD_MASTER_FILE = "CFD_Master_20260911.csv"

# UFD / uncertain development
UFD_ROUTING_FILE = "UncertainDevRoutingExample_0909_Remove_Dup.csv"
UFD_TC_FILE      = "MONTHLY_FORECAST_6COL.csv"

# Backward-compatible aliases used by the original downstream notebook
CFD_ROUTING_FILE = CFD_MASTER_FILE
UNCERTAIN_ROUTING_FILE = UFD_ROUTING_FILE
UNCERTAIN_TC_FILE = UFD_TC_FILE

MAX_WELL_MONTHS = 500
ANNOTATION_DATE = pd.Timestamp("2034-02-01")
CHUNK_SIZE = 250_000

# Horizontal CFD files: 6 data columns + 1 spacer
CFD_BLOCK_WIDTH = 7
CFD_DATA_START_ROW = 4
CFD_DATE_OFFSET = 0
CFD_OIL_OFFSET = 2
CFD_GAS_OFFSET = 4

SAVE_PNGS = True
PLOT_FOLDER = "Facility_Wedge_Plots"
if SAVE_PNGS:
    os.makedirs(PLOT_FOLDER, exist_ok=True)
print("Settings loaded")


In [ ]:
def normalize_facility(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).strip().upper().split())

def normalize_name(v):
    if pd.isna(v): return ""
    return re.sub(r"\s+", " ", str(v).strip()).upper()

def normalize_tc(v):
    if pd.isna(v): return ""
    return re.sub(r"\s+", " ", str(v).strip()).upper()

def safe_filename(v):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(v).strip()).strip("_") or "Facility"

def nearest_first_of_month(v):
    if pd.isna(v): return pd.NaT
    dt=pd.Timestamp(v)
    this=pd.Timestamp(dt.year,dt.month,1)
    nxt=this+pd.DateOffset(months=1)
    return this if abs(dt-this) <= abs(nxt-dt) else nxt

def wi_decimal(s,label):
    x=pd.to_numeric(s,errors="coerce")
    if x.isna().any(): raise ValueError(f"{label}: blank/non-numeric WI values found.")
    if ((x<0)|(x>100)).any(): raise ValueError(f"{label}: WI must be entered as 0-100 percent.")
    return x/100.0

def parse_cfd_dates(series):
    out=[]
    for v in series:
        if pd.isna(v) or str(v).strip()=="":
            out.append(pd.NaT); continue
        text=str(v).strip(); dt=pd.NaT
        for fmt in ("%y-%b","%b-%y"):
            if pd.isna(dt):
                try:
                    c=pd.to_datetime(text,format=fmt,errors="raise")
                    if 2000 <= c.year <= 2200: dt=c
                except: pass
        if pd.isna(dt):
            try:
                c=pd.to_datetime(text,errors="raise")
                if 2000 <= c.year <= 2200: dt=c
            except: pass
        if pd.isna(dt):
            try:
                n=float(v)
                if 30000 <= n <= 150000:
                    c=pd.Timestamp("1899-12-30")+pd.Timedelta(days=n)
                    if 2000 <= c.year <= 2200: dt=c
            except: pass
        if not pd.isna(dt): dt=pd.Timestamp(dt.year,dt.month,1)
        out.append(dt)
    return pd.Series(out,index=series.index,dtype="datetime64[ns]")


In [ ]:
# ============================================================
# ROUTING FILES = HARD WHITELISTS
# ============================================================

# ---------------- PDP ----------------
p = pd.read_csv(PDP_ROUTING_FILE, encoding="latin1")
if p.shape[1] < 5:
    raise ValueError("PDP routing needs columns A-E")
pdp_route = p.iloc[:, [0,1,4]].copy()
pdp_route.columns = ["Facility","Well_Name","WI_Pct"]
pdp_route = pdp_route.dropna(subset=["Facility","Well_Name","WI_Pct"], how="all").copy()
pdp_route["Facility"] = pdp_route["Facility"].apply(normalize_facility)
pdp_route["Well_Name"] = pdp_route["Well_Name"].astype(str).str.strip()
pdp_route["Match_Name"] = pdp_route["Well_Name"].apply(normalize_name)
pdp_route["WI"] = wi_decimal(pdp_route["WI_Pct"], "PDP routing")
pdp_route = pdp_route[(pdp_route["Facility"]!="") & (pdp_route["Match_Name"]!="")].copy()

# ---------------- CFD MASTER ----------------
c = pd.read_csv(CFD_MASTER_FILE, encoding="latin1")
if c.shape[1] < 7:
    raise ValueError("CFD Master needs columns A-G")
cfd_route = c.iloc[:, :7].copy()
cfd_route.columns = [
    "Facility", "Well_Name", "WI_Pct", "Production_Method",
    "POP_Date", "Forecast_File", "Production_Link"
]
# Ignore truly empty trailing CSV rows, but DO NOT silently remove a populated row with blank WI.
cfd_route = cfd_route.dropna(
    subset=["Facility","Well_Name","WI_Pct","Production_Method","POP_Date","Forecast_File","Production_Link"],
    how="all"
).copy()
cfd_route["Facility"] = cfd_route["Facility"].apply(normalize_facility)
cfd_route["Well_Name"] = cfd_route["Well_Name"].astype(str).str.strip()
cfd_route["Match_Name"] = cfd_route["Well_Name"].apply(normalize_name)
cfd_route["WI"] = wi_decimal(cfd_route["WI_Pct"], "CFD routing")
cfd_route["Production_Method"] = cfd_route["Production_Method"].fillna("").astype(str).str.strip().str.upper()
cfd_route["Forecast_File"] = cfd_route["Forecast_File"].fillna("").astype(str).str.strip()
cfd_route["Forecast_File_Key"] = cfd_route["Forecast_File"].apply(lambda x: os.path.basename(str(x)).strip().upper())
cfd_route["POP_Date"] = pd.to_datetime(cfd_route["POP_Date"], errors="coerce")
cfd_route["Normalized_POP"] = cfd_route["POP_Date"].apply(nearest_first_of_month)
cfd_route = cfd_route[(cfd_route["Facility"]!="") & (cfd_route["Match_Name"]!="")].copy()
if cfd_route["Forecast_File_Key"].eq("").any():
    display(cfd_route.loc[cfd_route["Forecast_File_Key"].eq(""), ["Facility","Well_Name","Forecast_File"]])
    raise ValueError("CFD Master has routed wells with blank Forecast CSV Location")

# ---------------- UFD ----------------
u = pd.read_csv(UFD_ROUTING_FILE, encoding="latin1")
if u.shape[1] < 5:
    raise ValueError("UFD routing needs columns A-E")
ud_route = u.iloc[:, [0,1,2,3,4]].copy()
ud_route.columns = ["Facility","Well_Name","WI_Pct","TC_Connection","POP_Date"]
# Remove only completely empty trailing rows. A populated row with blank WI remains an error.
ud_route = ud_route.dropna(
    subset=["Facility","Well_Name","WI_Pct","TC_Connection","POP_Date"], how="all"
).copy()
ud_route["Facility"] = ud_route["Facility"].apply(normalize_facility)
ud_route["Well_Name"] = ud_route["Well_Name"].astype(str).str.strip()
ud_route["Match_Name"] = ud_route["Well_Name"].apply(normalize_name)
ud_route["TC_Key"] = ud_route["TC_Connection"].apply(normalize_tc)
ud_route["WI"] = wi_decimal(ud_route["WI_Pct"], "UFD routing")
ud_route["POP_Date"] = pd.to_datetime(ud_route["POP_Date"], errors="coerce")
ud_route["Normalized_POP"] = ud_route["POP_Date"].apply(nearest_first_of_month)
ud_route = ud_route[(ud_route["Facility"]!="") & (ud_route["Match_Name"]!="")].copy()
if ud_route["TC_Key"].eq("").any() or ud_route["Normalized_POP"].isna().any():
    raise ValueError("UFD routing has blank TC or invalid POP date")

# ---------------- uniqueness checks ----------------
def check_unique(route, extra_cols, label):
    agg = {"Facility":"nunique", "WI":"nunique"}
    for x in extra_cols:
        agg[x] = "nunique"
    chk = route.groupby("Match_Name").agg(agg)
    if (chk > 1).any(axis=1).any():
        display(chk[(chk > 1).any(axis=1)])
        raise ValueError(f"{label}: duplicated well has conflicting routing")

check_unique(pdp_route, [], "PDP")
check_unique(cfd_route, ["Forecast_File_Key"], "CFD")
check_unique(ud_route, ["TC_Key","Normalized_POP"], "UFD")

pdp_route = pdp_route.drop_duplicates("Match_Name").reset_index(drop=True)
cfd_route = cfd_route.drop_duplicates("Match_Name").reset_index(drop=True)
ud_route = ud_route.drop_duplicates("Match_Name").reset_index(drop=True)

PDP_ALLOWED = set(pdp_route["Match_Name"])
CFD_ALLOWED = set(cfd_route["Match_Name"])
UD_TCS = set(ud_route["TC_Key"])

print("PDP routed wells:", len(PDP_ALLOWED))
print("CFD routed wells:", len(CFD_ALLOWED))
print("UFD routed wells:", ud_route["Match_Name"].nunique())


In [ ]:
# LARGE PDP READER - only A/H/I/J, filtered in chunks
def read_large_pdp(filename):
    keep=[]
    for chunk in pd.read_csv(filename,usecols=[0,7,8,9],chunksize=CHUNK_SIZE):
        chunk.columns=["Well_Name","Date","Oil_BBL","Gas_MCF"]
        chunk["Match_Name"]=chunk["Well_Name"].apply(normalize_name)
        chunk=chunk[chunk["Match_Name"].isin(PDP_ALLOWED)].copy()
        if len(chunk)==0: continue
        chunk["Date"]=pd.to_datetime(chunk["Date"],errors="coerce").dt.to_period("M").dt.to_timestamp()
        chunk["Oil_BBL"]=pd.to_numeric(chunk["Oil_BBL"],errors="coerce").fillna(0)
        chunk["Gas_MCF"]=pd.to_numeric(chunk["Gas_MCF"],errors="coerce").fillna(0)
        chunk=chunk[chunk["Date"].notna()]
        keep.append(chunk.groupby(["Match_Name","Date"],as_index=False)[["Oil_BBL","Gas_MCF"]].sum())
    if not keep: return pd.DataFrame(columns=["Match_Name","Date","Oil_BBL","Gas_MCF"])
    d=pd.concat(keep,ignore_index=True)
    return d.groupby(["Match_Name","Date"],as_index=False)[["Oil_BBL","Gas_MCF"]].sum()

print("Loading PDP production..."); pdp_prod=read_large_pdp(PDP_PRODUCTION_FILE); gc.collect()
print("Loading PDP forecast..."); pdp_fcst=read_large_pdp(PDP_FORECAST_FILE); gc.collect()
print("PDP production wells used:",pdp_prod["Match_Name"].nunique())
print("PDP forecast wells used:",pdp_fcst["Match_Name"].nunique())

In [ ]:
# Show routed PDP wells missing from production and/or forecast files
prod_wells = set(pdp_prod["Match_Name"].unique())
fcst_wells = set(pdp_fcst["Match_Name"].unique())
routed_wells = set(pdp_route["Match_Name"].unique())

print("\nRouted wells NOT found in PDP production:")
print(sorted(routed_wells - prod_wells))

print("\nRouted wells NOT found in PDP forecast:")
print(sorted(routed_wells - fcst_wells))

In [ ]:
# BUILD 500-MONTH NET PDP WELL PROFILES
def build_pdp_profiles():
    rows=[]; missing=[]
    for _,r in pdp_route.iterrows():
        key,fac,well,wi=r["Match_Name"],r["Facility"],r["Well_Name"],r["WI"]
        p=pdp_prod[pdp_prod["Match_Name"]==key].sort_values("Date").copy()
        f=pdp_fcst[pdp_fcst["Match_Name"]==key].sort_values("Date").copy()
        if len(p)==0 and len(f)==0:
            missing.append(well); continue
        start=p["Date"].min() if len(p) else f["Date"].min()
        prof=pd.DataFrame({"Date":pd.date_range(start=start,periods=MAX_WELL_MONTHS,freq="MS")})
        last_actual=p["Date"].max() if len(p) else pd.NaT
        pp=p[["Date","Oil_BBL","Gas_MCF"]].rename(columns={"Oil_BBL":"A_Oil","Gas_MCF":"A_Gas"})
        ff=f[["Date","Oil_BBL","Gas_MCF"]].rename(columns={"Oil_BBL":"F_Oil","Gas_MCF":"F_Gas"})
        prof=prof.merge(pp,on="Date",how="left").merge(ff,on="Date",how="left").fillna(0)
        if len(p): actual=prof["Date"]<=last_actual
        else: actual=pd.Series(False,index=prof.index)
        forecast=~actual
        prof["Gross_Oil_BBL"]=0.0; prof["Gross_Gas_MCF"]=0.0
        prof.loc[actual,"Gross_Oil_BBL"]=prof.loc[actual,"A_Oil"]
        prof.loc[actual,"Gross_Gas_MCF"]=prof.loc[actual,"A_Gas"]
        prof.loc[forecast,"Gross_Oil_BBL"]=prof.loc[forecast,"F_Oil"]
        prof.loc[forecast,"Gross_Gas_MCF"]=prof.loc[forecast,"F_Gas"]
        prof["Net_Oil_BBL"]=prof["Gross_Oil_BBL"]*wi
        prof["Net_Gas_MCF"]=prof["Gross_Gas_MCF"]*wi
        prof["Facility"]=fac; prof["Well_Name"]=well; prof["Match_Name"]=key
        rows.append(prof[["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"]])
    if missing:
        print("WARNING routed PDP wells missing from both source files:")
        for w in missing: print(" ",w)
    if not rows: raise ValueError("No PDP profiles built")
    return pd.concat(rows,ignore_index=True)

def aggregate_component(d):
    if d is None or len(d)==0: return pd.DataFrame(columns=["Facility","Date","Net_Oil_BBL","Net_Gas_MCF"])
    return d.groupby(["Facility","Date"],as_index=False)[["Net_Oil_BBL","Net_Gas_MCF"]].sum().sort_values(["Facility","Date"])

pdp_wells=build_pdp_profiles(); pdp_fac=aggregate_component(pdp_wells); gc.collect()
print("PDP profiles built:",pdp_wells["Match_Name"].nunique())

In [ ]:
# ============================================================
# CONFIRMED DEV / CFD - NEW MASTER-DRIVEN FORECAST READER
# Supports both:
#   1) horizontal repeating 7-column well blocks with explicit dates
#   2) vertical 6-column ENTITY / FORECAST MONTH / OIL / rate / GAS / rate
# The Forecast_File column in CFD Master decides which CSV supplies each well.
# WI is applied at well-month level before facility aggregation.
# ============================================================

def _read_cfd_horizontal(filename, wanted):
    raw = pd.read_csv(filename, header=None, encoding="latin1")
    out = {}
    for start in range(0, raw.shape[1], CFD_BLOCK_WIDTH):
        if start + CFD_GAS_OFFSET >= raw.shape[1]:
            continue
        v = raw.iloc[0, start]
        if pd.isna(v):
            continue
        key = normalize_name(v)
        if key not in wanted:
            continue
        t = pd.DataFrame({
            "Date": raw.iloc[CFD_DATA_START_ROW:, start + CFD_DATE_OFFSET].values,
            "Gross_Oil_BBL": raw.iloc[CFD_DATA_START_ROW:, start + CFD_OIL_OFFSET].values,
            "Gross_Gas_MCF": raw.iloc[CFD_DATA_START_ROW:, start + CFD_GAS_OFFSET].values,
        })
        t["Date"] = parse_cfd_dates(t["Date"])
        t["Gross_Oil_BBL"] = pd.to_numeric(t["Gross_Oil_BBL"], errors="coerce").fillna(0.0)
        t["Gross_Gas_MCF"] = pd.to_numeric(t["Gross_Gas_MCF"], errors="coerce").fillna(0.0)
        t = t[t["Date"].notna()].groupby("Date", as_index=False)[["Gross_Oil_BBL","Gross_Gas_MCF"]].sum().sort_values("Date")
        out[key] = t
    return out


def _read_cfd_vertical(filename, wanted):
    # Expected layout: A entity, B forecast month, C oil volume, D oil rate,
    # E gas volume, F gas rate. Header names are not required.
    d = pd.read_csv(filename, encoding="latin1")
    if d.shape[1] < 5:
        return {}
    x = d.iloc[:, [0,1,2,4]].copy()
    x.columns = ["Entity","Forecast_Month","Gross_Oil_BBL","Gross_Gas_MCF"]
    x["Match_Name"] = x["Entity"].apply(normalize_name)
    x = x[x["Match_Name"].isin(wanted)].copy()
    x["Forecast_Month"] = pd.to_numeric(x["Forecast_Month"], errors="coerce")
    x["Gross_Oil_BBL"] = pd.to_numeric(x["Gross_Oil_BBL"], errors="coerce").fillna(0.0)
    x["Gross_Gas_MCF"] = pd.to_numeric(x["Gross_Gas_MCF"], errors="coerce").fillna(0.0)
    x = x[x["Forecast_Month"].notna()].copy()
    x["Forecast_Month"] = x["Forecast_Month"].astype(int)
    x = x[(x["Forecast_Month"] >= 1) & (x["Forecast_Month"] <= MAX_WELL_MONTHS)]
    return {
        k: g.groupby("Forecast_Month", as_index=False)[["Gross_Oil_BBL","Gross_Gas_MCF"]].sum().sort_values("Forecast_Month")
        for k,g in x.groupby("Match_Name")
    }


def _read_cfd_source_auto(filename, wanted):
    # Try horizontal first because the legacy confirmed-dev files contain header-like
    # rows that are intentionally read with header=None. If no routed wells are found,
    # try the vertical 6-column forecast layout.
    h = _read_cfd_horizontal(filename, wanted)
    if h:
        return "HORIZONTAL", h
    v = _read_cfd_vertical(filename, wanted)
    if v:
        return "VERTICAL", v
    return "UNKNOWN", {}


def build_cfd_profiles():
    rows = []
    found = set()

    # Cache each distinct source file once.
    source_cache = {}
    for file_key, rr in cfd_route.groupby("Forecast_File_Key"):
        # Use the exact path/name entered in the master from the first matching row.
        filename = rr.iloc[0]["Forecast_File"]
        wanted = set(rr["Match_Name"])
        if not os.path.exists(filename):
            # Common JupyterLite case: master contains a path but uploaded file is in cwd.
            base = os.path.basename(filename)
            if os.path.exists(base):
                filename = base
            else:
                raise FileNotFoundError(f"CFD forecast file not found: {rr.iloc[0]['Forecast_File']}")
        layout, data = _read_cfd_source_auto(filename, wanted)
        source_cache[file_key] = (layout, data, filename)
        print(f"CFD source: {filename} -> {layout}, matched wells: {len(data)}")

    for _, r in cfd_route.iterrows():
        key = r["Match_Name"]
        layout, data, filename = source_cache[r["Forecast_File_Key"]]
        if key not in data:
            continue
        g = data[key].copy()

        if layout == "VERTICAL":
            if pd.isna(r["Normalized_POP"]):
                raise ValueError(f"CFD well {r['Well_Name']} uses forecast months but has no valid POP Date")
            g["Date"] = g["Forecast_Month"].apply(
                lambda m: r["Normalized_POP"] + pd.DateOffset(months=int(m)-1)
            )
        else:
            g["Date"] = pd.to_datetime(g["Date"], errors="coerce").dt.to_period("M").dt.to_timestamp()
            g = g[g["Date"].notna()].copy()
            nz = g[(g["Gross_Oil_BBL"] != 0) | (g["Gross_Gas_MCF"] != 0)]
            if len(nz):
                first = nz["Date"].min()
                end = first + pd.DateOffset(months=MAX_WELL_MONTHS-1)
                g = g[(g["Date"] >= first) & (g["Date"] <= end)].copy()

        g["Net_Oil_BBL"] = pd.to_numeric(g["Gross_Oil_BBL"], errors="coerce").fillna(0.0) * r["WI"]
        g["Net_Gas_MCF"] = pd.to_numeric(g["Gross_Gas_MCF"], errors="coerce").fillna(0.0) * r["WI"]
        g["Facility"] = r["Facility"]
        g["Well_Name"] = r["Well_Name"]
        g["Match_Name"] = key
        rows.append(g[["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"]])
        found.add(key)

    missing = CFD_ALLOWED - found
    if missing:
        print("WARNING routed CFD wells not found in their CFD forecast source:")
        lookup = cfd_route.set_index("Match_Name")
        for k in sorted(missing):
            print(" ", lookup.loc[k,"Well_Name"], "->", lookup.loc[k,"Forecast_File"])

    if not rows:
        return pd.DataFrame(columns=["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"])
    return pd.concat(rows, ignore_index=True)

cfd_wells = build_cfd_profiles()
cfd_fac = aggregate_component(cfd_wells)
gc.collect()
print("CFD profiles built:", cfd_wells["Match_Name"].nunique())


In [ ]:
# UNCERTAIN DEV - TC monthly forecasts + routed WI/POP/facility
def read_tc_forecasts():
    keep=[]
    for chunk in pd.read_csv(UNCERTAIN_TC_FILE,usecols=[0,1,2,4],chunksize=CHUNK_SIZE):
        chunk.columns=["TC_Connection","Forecast_Month","Gross_Oil_BBL","Gross_Gas_MCF"]
        chunk["TC_Key"]=chunk["TC_Connection"].apply(normalize_tc)
        chunk=chunk[chunk["TC_Key"].isin(UD_TCS)].copy()
        if len(chunk)==0: continue
        chunk["Forecast_Month"]=pd.to_numeric(chunk["Forecast_Month"],errors="coerce")
        chunk["Gross_Oil_BBL"]=pd.to_numeric(chunk["Gross_Oil_BBL"],errors="coerce").fillna(0)
        chunk["Gross_Gas_MCF"]=pd.to_numeric(chunk["Gross_Gas_MCF"],errors="coerce").fillna(0)
        chunk=chunk[chunk["Forecast_Month"].notna()].copy(); chunk["Forecast_Month"]=chunk["Forecast_Month"].astype(int)
        chunk=chunk[(chunk["Forecast_Month"]>=1)&(chunk["Forecast_Month"]<=MAX_WELL_MONTHS)]
        keep.append(chunk.groupby(["TC_Key","Forecast_Month"],as_index=False)[["Gross_Oil_BBL","Gross_Gas_MCF"]].sum())
    if not keep: return pd.DataFrame(columns=["TC_Key","Forecast_Month","Gross_Oil_BBL","Gross_Gas_MCF"])
    d=pd.concat(keep,ignore_index=True)
    return d.groupby(["TC_Key","Forecast_Month"],as_index=False)[["Gross_Oil_BBL","Gross_Gas_MCF"]].sum()

def build_uncertain_profiles(tc):
    groups={k:g.sort_values("Forecast_Month").copy() for k,g in tc.groupby("TC_Key")}
    rows=[]; missing=[]
    for _,r in ud_route.iterrows():
        if r["TC_Key"] not in groups:
            missing.append((r["Well_Name"],r["TC_Connection"])); continue
        g=groups[r["TC_Key"]].copy()
        g["Date"]=g["Forecast_Month"].apply(lambda m:r["Normalized_POP"]+pd.DateOffset(months=int(m)-1))
        g["Net_Oil_BBL"]=g["Gross_Oil_BBL"]*r["WI"]
        g["Net_Gas_MCF"]=g["Gross_Gas_MCF"]*r["WI"]
        g["Facility"]=r["Facility"]; g["Well_Name"]=r["Well_Name"]; g["Match_Name"]=r["Match_Name"]
        rows.append(g[["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"]])
    if missing:
        print("WARNING uncertain wells skipped because TC missing:")
        for w,tcname in missing: print(" ",w,"->",tcname)
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame(columns=["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"])

tc=read_tc_forecasts(); ud_wells=build_uncertain_profiles(tc); ud_fac=aggregate_component(ud_wells); gc.collect()
print("Uncertain profiles built:",ud_wells["Match_Name"].nunique())

In [ ]:
# BUILD THREE FACILITY-MONTH CASES
def merge_component(base,comp,prefix):
    c=comp.rename(columns={"Net_Oil_BBL":f"{prefix}_Oil_BBL","Net_Gas_MCF":f"{prefix}_Gas_MCF"})
    return base.merge(c,on=["Facility","Date"],how="outer")

def complete_calendar(d,cols):
    out=[]
    for fac,g in d.groupby("Facility"):
        cal=pd.DataFrame({"Date":pd.date_range(g["Date"].min(),g["Date"].max(),freq="MS")}); cal["Facility"]=fac
        cal=cal.merge(g,on=["Facility","Date"],how="left")
        for c in cols:
            if c not in cal: cal[c]=0.0
            cal[c]=pd.to_numeric(cal[c],errors="coerce").fillna(0)
        out.append(cal)
    return pd.concat(out,ignore_index=True)

def calc(d):
    d=d.sort_values(["Facility","Date"]).reset_index(drop=True).copy()
    d["Days_In_Month"]=d["Date"].dt.days_in_month
    d["Total_Oil_BPD"]=d["Total_Oil_BBL"]/d["Days_In_Month"]
    d["Total_Gas_MCFD"]=d["Total_Gas_MCF"]/d["Days_In_Month"]
    d["Total_Oil_Cumulative_BBL"]=d.groupby("Facility")["Total_Oil_BBL"].cumsum()
    d["Total_Gas_Cumulative_MCF"]=d.groupby("Facility")["Total_Gas_MCF"].cumsum()
    return d

case1=pdp_fac.rename(columns={"Net_Oil_BBL":"PDP_Oil_BBL","Net_Gas_MCF":"PDP_Gas_MCF"}).copy()
case1=complete_calendar(case1,["PDP_Oil_BBL","PDP_Gas_MCF"]); case1["Total_Oil_BBL"]=case1["PDP_Oil_BBL"]; case1["Total_Gas_MCF"]=case1["PDP_Gas_MCF"]; case1=calc(case1)

case2=merge_component(case1[["Facility","Date","PDP_Oil_BBL","PDP_Gas_MCF"]],cfd_fac,"Confirmed")
case2=complete_calendar(case2,["PDP_Oil_BBL","PDP_Gas_MCF","Confirmed_Oil_BBL","Confirmed_Gas_MCF"]); case2["Total_Oil_BBL"]=case2["PDP_Oil_BBL"]+case2["Confirmed_Oil_BBL"]; case2["Total_Gas_MCF"]=case2["PDP_Gas_MCF"]+case2["Confirmed_Gas_MCF"]; case2=calc(case2)

case3=merge_component(case2[["Facility","Date","PDP_Oil_BBL","PDP_Gas_MCF","Confirmed_Oil_BBL","Confirmed_Gas_MCF"]],ud_fac,"Uncertain")
case3=complete_calendar(case3,["PDP_Oil_BBL","PDP_Gas_MCF","Confirmed_Oil_BBL","Confirmed_Gas_MCF","Uncertain_Oil_BBL","Uncertain_Gas_MCF"]); case3["Total_Oil_BBL"]=case3["PDP_Oil_BBL"]+case3["Confirmed_Oil_BBL"]+case3["Uncertain_Oil_BBL"]; case3["Total_Gas_MCF"]=case3["PDP_Gas_MCF"]+case3["Confirmed_Gas_MCF"]+case3["Uncertain_Gas_MCF"]; case3=calc(case3)

# Add prior-case columns for wedge plots
def add_prior(total,prior):
    p=prior[["Facility","Date","Total_Oil_BPD","Total_Gas_MCFD","Total_Oil_Cumulative_BBL","Total_Gas_Cumulative_MCF"]].rename(columns={"Total_Oil_BPD":"Prior_Oil_BPD","Total_Gas_MCFD":"Prior_Gas_MCFD","Total_Oil_Cumulative_BBL":"Prior_Oil_Cumulative_BBL","Total_Gas_Cumulative_MCF":"Prior_Gas_Cumulative_MCF"})
    x=total.merge(p,on=["Facility","Date"],how="left")
    for c in ["Prior_Oil_BPD","Prior_Gas_MCFD","Prior_Oil_Cumulative_BBL","Prior_Gas_Cumulative_MCF"]: x[c]=x[c].fillna(0)
    return x
case2=add_prior(case2,case1); case3=add_prior(case3,case2)
print("Cases built")

In [ ]:
# PLOTS
def plot_profile(d,facility,label,stream,prior_label=None):
    g=d[d["Facility"]==facility].sort_values("Date").copy()
    if len(g)==0:return
    if stream=="Oil": rate,cum="Total_Oil_BPD","Total_Oil_Cumulative_BBL"; prate,pcum="Prior_Oil_BPD","Prior_Oil_Cumulative_BBL"; runit,cunit="BPD","BBL"
    else: rate,cum="Total_Gas_MCFD","Total_Gas_Cumulative_MCF"; prate,pcum="Prior_Gas_MCFD","Prior_Gas_Cumulative_MCF"; runit,cunit="MCF/D","MCF"
    x=list(g["Date"].dt.to_pydatetime()); y=g[rate].to_numpy(float); c=g[cum].to_numpy(float)
    peak=float(y.max()); row=g[g["Date"]==ANNOTATION_DATE]; r2034=float(row.iloc[0][rate]) if len(row) else np.nan; end=float(c[-1])
    fig,ax1=plt.subplots(figsize=(14,7)); ax2=ax1.twinx(); handles=[]; labels=[]
    if prior_label is None:
        l1,=ax1.plot(x,y,linewidth=2); l2,=ax2.plot(x,c,"--",linewidth=2); handles=[l1,l2]; labels=[f"{label} Rate",f"{label} Cumulative"]
    else:
        py=g[prate].to_numpy(float); pc=g[pcum].to_numpy(float)
        l0,=ax1.plot(x,py,linewidth=1.5); l1,=ax1.plot(x,y,linewidth=2); ax1.fill_between(x,py,y,alpha=.22)
        l2,=ax2.plot(x,pc,"--",linewidth=1.5); l3,=ax2.plot(x,c,"--",linewidth=2)
        handles=[l0,l1,l2,l3]; labels=[f"{prior_label} Rate",f"{label} Rate",f"{prior_label} Cumulative",f"{label} Cumulative"]
    rtxt=f"{r2034:,.0f} {runit}" if np.isfinite(r2034) else "N/A"
    ax1.text(.015,.97,f"Peak Rate: {peak:,.0f} {runit}\nRate on 2/1/2034: {rtxt}\nEnd Cumulative: {end:,.0f} {cunit}",transform=ax1.transAxes,va="top",bbox=dict(boxstyle="round",facecolor="white",alpha=.88))
    ax1.set_title(f"{facility} - {stream} - {label}"); ax1.set_xlabel("Date"); ax1.set_ylabel(f"{stream} Rate ({runit})"); ax2.set_ylabel(f"Cumulative {stream} ({cunit})"); ax1.grid(True,alpha=.3); ax1.legend(handles,labels,loc="best")
    if SAVE_PNGS: fig.savefig(os.path.join(PLOT_FOLDER,f"{safe_filename(label)}_{safe_filename(facility)}_{stream}.png"),dpi=160,bbox_inches="tight")
    plt.show()

for fac in sorted(case1["Facility"].dropna().unique()):
    plot_profile(case1,fac,"PDP Base","Oil"); plot_profile(case1,fac,"PDP Base","Gas")

cfd_affected=set(cfd_fac.loc[(cfd_fac["Net_Oil_BBL"]!=0)|(cfd_fac["Net_Gas_MCF"]!=0),"Facility"])
for fac in sorted(cfd_affected):
    plot_profile(case2,fac,"PDP + Confirmed Dev","Oil","PDP Base"); plot_profile(case2,fac,"PDP + Confirmed Dev","Gas","PDP Base")

ud_affected=set(ud_fac.loc[(ud_fac["Net_Oil_BBL"]!=0)|(ud_fac["Net_Gas_MCF"]!=0),"Facility"])
for fac in sorted(ud_affected):
    plot_profile(case3,fac,"PDP + Confirmed + Uncertain Dev","Oil","PDP + Confirmed Dev"); plot_profile(case3,fac,"PDP + Confirmed + Uncertain Dev","Gas","PDP + Confirmed Dev")

In [ ]:
# ============================================================
# EXPORT 01-05 WITH HERCULES RULE
# OIL: CPF + EAST SATELLITE + WEST SATELLITE -> HERCULES
# GAS: keep the three Hercules facilities separate
# This export mapping does not overwrite the in-memory case1/case2/case3
# objects used by the original downstream plotting/ranking/sensitivity code.
# ============================================================
HERCULES_COMPONENTS = {
    "HERCULES CPF",
    "HERCULES EAST SATELLITE",
    "HERCULES WEST SATELLITE",
}

def _export_facility(v, stream):
    f = normalize_facility(v)
    if stream == "Oil" and f in HERCULES_COMPONENTS:
        return "HERCULES"
    return f


def _stream_export(d, name, stream):
    x = d.copy()
    x["Facility"] = x["Facility"].apply(lambda v: _export_facility(v, stream))
    x["Date"] = pd.to_datetime(x["Date"], errors="coerce").dt.to_period("M").dt.to_timestamp()

    if stream == "Oil":
        comp = [c for c in ["PDP_Oil_BBL","Confirmed_Oil_BBL","Uncertain_Oil_BBL"] if c in x.columns]
        keep = ["Facility","Date"] + comp
        x = x[keep].copy()
        for c in comp: x[c] = pd.to_numeric(x[c], errors="coerce").fillna(0.0)
        x = x.groupby(["Facility","Date"], as_index=False)[comp].sum()
        x["Total_Oil_BBL"] = x[comp].sum(axis=1)
        x["Total_Oil_BPD"] = x["Total_Oil_BBL"] / x["Date"].dt.days_in_month
        x = x.sort_values(["Facility","Date"])
        x["Total_Oil_Cumulative_BBL"] = x.groupby("Facility")["Total_Oil_BBL"].cumsum()
    else:
        comp = [c for c in ["PDP_Gas_MCF","Confirmed_Gas_MCF","Uncertain_Gas_MCF"] if c in x.columns]
        keep = ["Facility","Date"] + comp
        x = x[keep].copy()
        for c in comp: x[c] = pd.to_numeric(x[c], errors="coerce").fillna(0.0)
        x = x.groupby(["Facility","Date"], as_index=False)[comp].sum()
        x["Total_Gas_MCF"] = x[comp].sum(axis=1)
        x["Total_Gas_MCFD"] = x["Total_Gas_MCF"] / x["Date"].dt.days_in_month
        x = x.sort_values(["Facility","Date"])
        x["Total_Gas_Cumulative_MCF"] = x.groupby("Facility")["Total_Gas_MCF"].cumsum()

    x.insert(0, "Case", name)
    return x.reset_index(drop=True)


def _component_export(comp, name, stream):
    x = comp.copy()
    x["Facility"] = x["Facility"].apply(lambda v: _export_facility(v, stream))
    x["Date"] = pd.to_datetime(x["Date"], errors="coerce").dt.to_period("M").dt.to_timestamp()
    if stream == "Oil":
        x["Total_Oil_BBL"] = pd.to_numeric(x["Net_Oil_BBL"], errors="coerce").fillna(0.0)
        x = x.groupby(["Facility","Date"], as_index=False)["Total_Oil_BBL"].sum().sort_values(["Facility","Date"])
        x["Total_Oil_BPD"] = x["Total_Oil_BBL"] / x["Date"].dt.days_in_month
        x["Total_Oil_Cumulative_BBL"] = x.groupby("Facility")["Total_Oil_BBL"].cumsum()
    else:
        x["Total_Gas_MCF"] = pd.to_numeric(x["Net_Gas_MCF"], errors="coerce").fillna(0.0)
        x = x.groupby(["Facility","Date"], as_index=False)["Total_Gas_MCF"].sum().sort_values(["Facility","Date"])
        x["Total_Gas_MCFD"] = x["Total_Gas_MCF"] / x["Date"].dt.days_in_month
        x["Total_Gas_Cumulative_MCF"] = x.groupby("Facility")["Total_Gas_MCF"].cumsum()
    x.insert(0, "Case", name)
    return x.reset_index(drop=True)

# 01-03
_stream_export(case1, "PDP Base", "Oil").to_csv("01_PDP_Base_Oil.csv", index=False)
_stream_export(case1, "PDP Base", "Gas").to_csv("01_PDP_Base_Gas.csv", index=False)
_stream_export(case2, "PDP + Confirmed Dev", "Oil").to_csv("02_PDP_Plus_Confirmed_Oil.csv", index=False)
_stream_export(case2, "PDP + Confirmed Dev", "Gas").to_csv("02_PDP_Plus_Confirmed_Gas.csv", index=False)
_stream_export(case3, "PDP + Confirmed + Uncertain Dev", "Oil").to_csv("03_PDP_Plus_Confirmed_Plus_Uncertain_Oil.csv", index=False)
_stream_export(case3, "PDP + Confirmed + Uncertain Dev", "Gas").to_csv("03_PDP_Plus_Confirmed_Plus_Uncertain_Gas.csv", index=False)

# 04 CFD-only / 05 UFD-only QA outputs
_component_export(cfd_fac, "CFD Only", "Oil").to_csv("04_CFD_Vols_Oil.csv", index=False)
_component_export(cfd_fac, "CFD Only", "Gas").to_csv("04_CFD_Vols_Gas.csv", index=False)
_component_export(ud_fac, "UFD Only", "Oil").to_csv("05_UFD_Vols_Oil.csv", index=False)
_component_export(ud_fac, "UFD Only", "Gas").to_csv("05_UFD_Vols_Gas.csv", index=False)

print("Saved 01-05 oil/gas CSV outputs")
print("Hercules oil is combined; Hercules gas remains split in all exported files.")


In [ ]:
# FINAL ON-SCREEN QA SUMMARY (not saved as an extra CSV)
def qa(d,name):
    rows=[]
    for fac,g in d.groupby("Facility"):
        g=g.sort_values("Date"); oi=g["Total_Oil_BPD"].idxmax(); gi=g["Total_Gas_MCFD"].idxmax(); r=g[g["Date"]==ANNOTATION_DATE]
        rows.append({"Case":name,"Facility":fac,"Peak_Oil_BPD":g.loc[oi,"Total_Oil_BPD"],"Peak_Oil_Date":g.loc[oi,"Date"],"Peak_Gas_MCFD":g.loc[gi,"Total_Gas_MCFD"],"Peak_Gas_Date":g.loc[gi,"Date"],"Oil_BPD_2_1_2034":float(r.iloc[0]["Total_Oil_BPD"]) if len(r) else np.nan,"Gas_MCFD_2_1_2034":float(r.iloc[0]["Total_Gas_MCFD"]) if len(r) else np.nan,"End_Oil_Cum_BBL":g.iloc[-1]["Total_Oil_Cumulative_BBL"],"End_Gas_Cum_MCF":g.iloc[-1]["Total_Gas_Cumulative_MCF"]})
    return pd.DataFrame(rows)
qa_summary=pd.concat([qa(case1,"PDP Base"),qa(case2,"PDP + Confirmed Dev"),qa(case3,"PDP + Confirmed + Uncertain Dev")],ignore_index=True)
display(qa_summary)
print("500-month well-life cap applied to PDP, CFD, and uncertain dev")
print("All output volumes are net of WI")
print("Routing files are hard whitelists")

# add columns that capture peak rate and date post-2034
# and have the well count 
# try to do rate annualization as well for 2027 and 2034 with corresponding well counts
# include output of annualized rates by year now through 2036 by each facility

In [ ]:
# ============================================================
# REGENERATE ALL PLOTS WITH:
# 1) POST-2034 PEAK RATE + DATE
# 2) TRUNCATE PLOT WHEN PDP DATA ENDS
# ============================================================
POST_2034_START = pd.Timestamp("2034-01-01")
def last_nonzero_pdp_date(df, facility):
    d = df[df["Facility"] == facility].copy()
    # Use whichever PDP columns exist in this case
    oil_col = "PDP_Oil_BBL" if "PDP_Oil_BBL" in d.columns else None
    gas_col = "PDP_Gas_MCF" if "PDP_Gas_MCF" in d.columns else None
    if oil_col is None and gas_col is None:
        return d["Date"].max()
    mask = pd.Series(False, index=d.index)
    if oil_col is not None:
        mask = mask | (d[oil_col].fillna(0) != 0)
    if gas_col is not None:
        mask = mask | (d[gas_col].fillna(0) != 0)
    if mask.any():
        return d.loc[mask, "Date"].max()
    return d["Date"].max()
def plot_facility_profile_v2(
    df,
    facility,
    case_label,
    stream,
    prior_case_label=None
):
    d = (
        df[df["Facility"] == facility]
        .sort_values("Date")
        .copy()
    )
    if len(d) == 0:
        return
    # --------------------------------------------------------
    # TRUNCATE AT LAST PDP MONTH WITH NONZERO DATA
    # --------------------------------------------------------
    pdp_end = last_nonzero_pdp_date(df, facility)
    d = d[
        d["Date"] <= pdp_end
    ].copy()
    if len(d) == 0:
        return
    # --------------------------------------------------------
    # STREAM COLUMN SETUP
    # --------------------------------------------------------
    if stream == "Oil":
        rate_col = "Total_Oil_BPD"
        cum_col = "Total_Oil_Cumulative_BBL"
        prior_rate_col = "Prior_Oil_BPD"
        prior_cum_col = "Prior_Oil_Cumulative_BBL"
        rate_unit = "BPD"
        cum_unit = "BBL"
    else:
        rate_col = "Total_Gas_MCFD"
        cum_col = "Total_Gas_Cumulative_MCF"
        prior_rate_col = "Prior_Gas_MCFD"
        prior_cum_col = "Prior_Gas_Cumulative_MCF"
        rate_unit = "MCF/D"
        cum_unit = "MCF"
    x = list(
        d["Date"].dt.to_pydatetime()
    )
    rate = d[rate_col].to_numpy(dtype=float)
    cumulative = d[cum_col].to_numpy(dtype=float)
    # --------------------------------------------------------
    # OVERALL PEAK
    # --------------------------------------------------------
    peak_idx = d[rate_col].idxmax()
    peak_rate = d.loc[
        peak_idx,
        rate_col
    ]
    peak_date = d.loc[
        peak_idx,
        "Date"
    ]
    # --------------------------------------------------------
    # RATE ON 2/1/2034
    # --------------------------------------------------------
    row_2034 = d[
        d["Date"] == pd.Timestamp("2034-02-01")
    ]
    if len(row_2034) > 0:
        rate_2034 = float(
            row_2034.iloc[0][rate_col]
        )
        rate_2034_text = f"{rate_2034:,.0f} {rate_unit}"
    else:
        rate_2034_text = "N/A"
    # --------------------------------------------------------
    # POST-2034 PEAK + DATE
    # --------------------------------------------------------
    post = d[
        d["Date"] >= POST_2034_START
    ].copy()
    if len(post) > 0:
        post_peak_idx = post[
            rate_col
        ].idxmax()
        post_peak_rate = post.loc[
            post_peak_idx,
            rate_col
        ]
        post_peak_date = post.loc[
            post_peak_idx,
            "Date"
        ]
        post_peak_text = (
            f"{post_peak_rate:,.0f} {rate_unit} "
            f"on {post_peak_date.strftime('%m/%d/%Y')}"
        )
    else:
        post_peak_text = "N/A"
    # --------------------------------------------------------
    # END CUMULATIVE
    # --------------------------------------------------------
    end_cumulative = float(
        cumulative[-1]
    )
    # --------------------------------------------------------
    # PLOT
    # --------------------------------------------------------
    fig, ax1 = plt.subplots(
        figsize=(14, 7)
    )
    ax2 = ax1.twinx()
    if prior_case_label is None:
        rate_line, = ax1.plot(
            x,
            rate,
            linewidth=2,
            label=f"{case_label} Rate"
        )
        cum_line, = ax2.plot(
            x,
            cumulative,
            linestyle="--",
            linewidth=2,
            label=f"{case_label} Cumulative"
        )
        legend_handles = [
            rate_line,
            cum_line
        ]
        legend_labels = [
            f"{case_label} Rate",
            f"{case_label} Cumulative"
        ]
    else:
        prior_rate = (
            d[prior_rate_col]
            .to_numpy(dtype=float)
        )
        prior_cum = (
            d[prior_cum_col]
            .to_numpy(dtype=float)
        )
        prior_rate_line, = ax1.plot(
            x,
            prior_rate,
            linewidth=1.6,
            label=f"{prior_case_label} Rate"
        )
        total_rate_line, = ax1.plot(
            x,
            rate,
            linewidth=2,
            label=f"{case_label} Rate"
        )
        ax1.fill_between(
            x,
            prior_rate,
            rate,
            alpha=0.22,
            label="Incremental Wedge"
        )
        prior_cum_line, = ax2.plot(
            x,
            prior_cum,
            linestyle="--",
            linewidth=1.6,
            label=f"{prior_case_label} Cumulative"
        )
        total_cum_line, = ax2.plot(
            x,
            cumulative,
            linestyle="--",
            linewidth=2,
            label=f"{case_label} Cumulative"
        )
        legend_handles = [
            prior_rate_line,
            total_rate_line,
            prior_cum_line,
            total_cum_line
        ]
        legend_labels = [
            f"{prior_case_label} Rate",
            f"{case_label} Rate",
            f"{prior_case_label} Cumulative",
            f"{case_label} Cumulative"
        ]
    # --------------------------------------------------------
    # NEW ANNOTATION
    # --------------------------------------------------------
    note = (
        f"Peak Rate: {peak_rate:,.0f} {rate_unit} "
        f"on {peak_date.strftime('%m/%d/%Y')}\n"
        f"Post-2034 Peak: {post_peak_text}\n"
        f"Rate on 2/1/2034: {rate_2034_text}\n"
        f"End Cumulative: {end_cumulative:,.0f} {cum_unit}"
    )
    ax1.text(
        0.015,
        0.97,
        note,
        transform=ax1.transAxes,
        va="top",
        ha="left",
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.88
        )
    )
    ax1.set_title(
        f"{facility} - {stream} - {case_label}"
    )
    ax1.set_xlabel("Date")
    ax1.set_ylabel(f"{stream} Rate ({rate_unit})")
    ax2.set_ylabel(f"Cumulative {stream} ({cum_unit})")
    ax1.grid(True, alpha=0.30)
    ax1.legend(
        legend_handles,
        legend_labels,
        loc="best"
    )
    plt.tight_layout()
    plt.show()
# ============================================================
# REGENERATE CASE 1
# PDP BASE
# ============================================================
for facility in sorted(
    case1["Facility"].dropna().unique()
):
    plot_facility_profile_v2(
        case1,
        facility,
        "PDP Base",
        "Oil"
    )
    plot_facility_profile_v2(
        case1,
        facility,
        "PDP Base",
        "Gas"
    )
# ============================================================
# REGENERATE CASE 2
# PDP + CONFIRMED DEV
# ============================================================
confirmed_affected = set(
    cfd_facility.loc[
        (cfd_facility["Net_Oil_BBL"] != 0) |
        (cfd_facility["Net_Gas_MCF"] != 0),
        "Facility"
    ]
)
for facility in sorted(confirmed_affected):
    plot_facility_profile_v2(
        case2,
        facility,
        "PDP + Confirmed Dev",
        "Oil",
        prior_case_label="PDP Base"
    )
    plot_facility_profile_v2(
        case2,
        facility,
        "PDP + Confirmed Dev",
        "Gas",
        prior_case_label="PDP Base"
    )
# ============================================================
# REGENERATE CASE 3
# PDP + CONFIRMED + UNCERTAIN DEV
# ============================================================
uncertain_affected = set(
    ud_facility.loc[
        (ud_facility["Net_Oil_BBL"] != 0) |
        (ud_facility["Net_Gas_MCF"] != 0),
        "Facility"
    ]
)
for facility in sorted(uncertain_affected):
    plot_facility_profile_v2(
        case3,
        facility,
        "PDP + Confirmed + Uncertain Dev",
        "Oil",
        prior_case_label="PDP + Confirmed Dev"
    )
    plot_facility_profile_v2(
        case3,
        facility,
        "PDP + Confirmed + Uncertain Dev",
        "Gas",
        prior_case_label="PDP + Confirmed Dev"
)




In [ ]:
# ============================================================
# FINAL RE-PLOT CELL
#
# Uses ONLY case1, case2, case3 from the working notebook.
#
# Adds:
#   - overall peak rate + date
#   - post-2034 peak rate + date
#   - rate on 2/1/2034
#   - end cumulative
#   - truncates plots at last nonzero PDP month
#
# Does NOT change any underlying calculations or CSV outputs.
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
POST_2034_START = pd.Timestamp("2034-01-01")
RATE_CHECK_DATE = pd.Timestamp("2034-02-01")
# ============================================================
# PREPARE CASE DATA
# Recalculates rate/cumulative locally just to make this
# plotting cell self-contained and robust.
# ============================================================
def prepare_case_for_plotting(df):
    d = df.copy()
    d["Date"] = pd.to_datetime(
        d["Date"],
        errors="coerce"
    )
    d = d[
        d["Date"].notna()
    ].copy()
    d = d.sort_values(
        ["Facility", "Date"]
    ).reset_index(drop=True)
    # Make sure total volume columns exist
    if "Total_Oil_BBL" not in d.columns:
        raise ValueError(
            "Total_Oil_BBL is missing from the case dataframe."
        )
    if "Total_Gas_MCF" not in d.columns:
        raise ValueError(
            "Total_Gas_MCF is missing from the case dataframe."
        )
    d["Total_Oil_BBL"] = pd.to_numeric(
        d["Total_Oil_BBL"],
        errors="coerce"
    ).fillna(0.0)
    d["Total_Gas_MCF"] = pd.to_numeric(
        d["Total_Gas_MCF"],
        errors="coerce"
    ).fillna(0.0)
    d["Days_In_Month"] = (
        d["Date"].dt.days_in_month
    )
    d["Plot_Oil_BPD"] = (
        d["Total_Oil_BBL"]
        /
        d["Days_In_Month"]
    )
    d["Plot_Gas_MCFD"] = (
        d["Total_Gas_MCF"]
        /
        d["Days_In_Month"]
    )
    d["Plot_Oil_Cum_BBL"] = (
        d.groupby("Facility")[
            "Total_Oil_BBL"
        ].cumsum()
    )
    d["Plot_Gas_Cum_MCF"] = (
        d.groupby("Facility")[
            "Total_Gas_MCF"
        ].cumsum()
    )
    return d
case1_plot = prepare_case_for_plotting(case1)
case2_plot = prepare_case_for_plotting(case2)
case3_plot = prepare_case_for_plotting(case3)
# ============================================================
# DETERMINE LAST REAL PDP MONTH
#
# Uses PDP oil/gas component columns.
# This prevents the graph from extending into artificial
# zero months at the end of the PDP profile.
# ============================================================
def get_last_pdp_date(df, facility):
    d = df[
        df["Facility"] == facility
    ].copy()
    if len(d) == 0:
        return pd.NaT
    oil_col = (
        "PDP_Oil_BBL"
        if "PDP_Oil_BBL" in d.columns
        else None
    )
    gas_col = (
        "PDP_Gas_MCF"
        if "PDP_Gas_MCF" in d.columns
        else None
    )
    # If for some reason PDP component columns are absent,
    # just use the dataframe end.
    if oil_col is None and gas_col is None:
        return d["Date"].max()
    active = pd.Series(
        False,
        index=d.index
    )
    if oil_col is not None:
        active = active | (
            pd.to_numeric(
                d[oil_col],
                errors="coerce"
            ).fillna(0.0) != 0
        )
    if gas_col is not None:
        active = active | (
            pd.to_numeric(
                d[gas_col],
                errors="coerce"
            ).fillna(0.0) != 0
        )
    if active.any():
        return d.loc[
            active,
            "Date"
        ].max()
    return d["Date"].max()
# ============================================================
# FIND FACILITIES ACTUALLY AFFECTED BY CONFIRMED DEV
#
# First tries component columns.
# If those do not exist, it derives affected facilities by
# comparing case2 total volumes against case1.
# ============================================================
def get_confirmed_affected():
    if (
        "Confirmed_Oil_BBL" in case2_plot.columns
        and
        "Confirmed_Gas_MCF" in case2_plot.columns
    ):
        mask = (
            pd.to_numeric(
                case2_plot["Confirmed_Oil_BBL"],
                errors="coerce"
            ).fillna(0.0) != 0
        ) | (
            pd.to_numeric(
                case2_plot["Confirmed_Gas_MCF"],
                errors="coerce"
            ).fillna(0.0) != 0
        )
        return set(
            case2_plot.loc[
                mask,
                "Facility"
            ].unique()
        )
    # Fallback: compare case 2 against case 1
    a = case1_plot[
        [
            "Facility",
            "Date",
            "Total_Oil_BBL",
            "Total_Gas_MCF"
        ]
    ].rename(
        columns={
            "Total_Oil_BBL": "Base_Oil",
            "Total_Gas_MCF": "Base_Gas"
        }
    )
    b = case2_plot[
        [
            "Facility",
            "Date",
            "Total_Oil_BBL",
            "Total_Gas_MCF"
        ]
    ]
    x = b.merge(
        a,
        on=["Facility", "Date"],
        how="left"
    )
    x["Base_Oil"] = x["Base_Oil"].fillna(0.0)
    x["Base_Gas"] = x["Base_Gas"].fillna(0.0)
    mask = (
        np.abs(
            x["Total_Oil_BBL"] - x["Base_Oil"]
        ) > 1e-9
    ) | (
        np.abs(
            x["Total_Gas_MCF"] - x["Base_Gas"]
        ) > 1e-9
    )
    return set(
        x.loc[
            mask,
            "Facility"
        ].unique()
    )
# ============================================================
# FIND FACILITIES ACTUALLY AFFECTED BY UNCERTAIN DEV
#
# First tries component columns.
# Otherwise compares case3 vs case2.
# ============================================================
def get_uncertain_affected():
    if (
        "Uncertain_Oil_BBL" in case3_plot.columns
        and
        "Uncertain_Gas_MCF" in case3_plot.columns
    ):
        mask = (
            pd.to_numeric(
                case3_plot["Uncertain_Oil_BBL"],
                errors="coerce"
            ).fillna(0.0) != 0
        ) | (
            pd.to_numeric(
                case3_plot["Uncertain_Gas_MCF"],
                errors="coerce"
            ).fillna(0.0) != 0
        )
        return set(
            case3_plot.loc[
                mask,
                "Facility"
            ].unique()
        )
    # Fallback: compare case 3 against case 2
    a = case2_plot[
        [
            "Facility",
            "Date",
            "Total_Oil_BBL",
            "Total_Gas_MCF"
        ]
    ].rename(
        columns={
            "Total_Oil_BBL": "Prior_Oil",
            "Total_Gas_MCF": "Prior_Gas"
        }
    )
    b = case3_plot[
        [
            "Facility",
            "Date",
            "Total_Oil_BBL",
            "Total_Gas_MCF"
        ]
    ]
    x = b.merge(
        a,
        on=["Facility", "Date"],
        how="left"
    )
    x["Prior_Oil"] = x["Prior_Oil"].fillna(0.0)
    x["Prior_Gas"] = x["Prior_Gas"].fillna(0.0)
    mask = (
        np.abs(
            x["Total_Oil_BBL"] - x["Prior_Oil"]
        ) > 1e-9
    ) | (
        np.abs(
            x["Total_Gas_MCF"] - x["Prior_Gas"]
        ) > 1e-9
    )
    return set(
        x.loc[
            mask,
            "Facility"
        ].unique()
    )
confirmed_affected = get_confirmed_affected()
uncertain_affected = get_uncertain_affected()
print(
    "Confirmed-development affected facilities:",
    len(confirmed_affected)
)
print(
    "Uncertain-development affected facilities:",
    len(uncertain_affected)
)
# ============================================================
# MAIN PLOT FUNCTION
# ============================================================
def plot_profile_final(
    current_df,
    facility,
    case_label,
    stream,
    prior_df=None,
    prior_label=None
):
    d = current_df[
        current_df["Facility"] == facility
    ].copy()
    if len(d) == 0:
        return
    # --------------------------------------------------------
    # TRUNCATE AT END OF REAL PDP DATA
    # --------------------------------------------------------
    pdp_end = get_last_pdp_date(
        current_df,
        facility
    )
    d = d[
        d["Date"] <= pdp_end
    ].copy()
    d = d.sort_values(
        "Date"
    )
    if len(d) == 0:
        return
    # --------------------------------------------------------
    # STREAM SETUP
    # --------------------------------------------------------
    if stream == "Oil":
        rate_col = "Plot_Oil_BPD"
        cum_col = "Plot_Oil_Cum_BBL"
        rate_unit = "BPD"
        cum_unit = "BBL"
    else:
        rate_col = "Plot_Gas_MCFD"
        cum_col = "Plot_Gas_Cum_MCF"
        rate_unit = "MCF/D"
        cum_unit = "MCF"
    # --------------------------------------------------------
    # OVERALL PEAK RATE + DATE
    # --------------------------------------------------------
    peak_idx = d[
        rate_col
    ].idxmax()
    peak_rate = float(
        d.loc[
            peak_idx,
            rate_col
        ]
    )
    peak_date = pd.Timestamp(
        d.loc[
            peak_idx,
            "Date"
        ]
    )
    # --------------------------------------------------------
    # RATE ON 2/1/2034
    # --------------------------------------------------------
    check = d[
        d["Date"] == RATE_CHECK_DATE
    ]
    if len(check) > 0:
        rate_check = float(
            check.iloc[0][rate_col]
        )
        rate_check_text = (
            f"{rate_check:,.0f} "
            f"{rate_unit}"
        )
    else:
        rate_check_text = "N/A"
    # --------------------------------------------------------
    # POST-2034 PEAK RATE + DATE
    # --------------------------------------------------------
    post = d[
        d["Date"] >= POST_2034_START
    ].copy()
    if len(post) > 0:
        post_idx = post[
            rate_col
        ].idxmax()
        post_peak_rate = float(
            post.loc[
                post_idx,
                rate_col
            ]
        )
        post_peak_date = pd.Timestamp(
            post.loc[
                post_idx,
                "Date"
            ]
        )
        post_peak_text = (
            f"{post_peak_rate:,.0f} "
            f"{rate_unit} on "
            f"{post_peak_date.strftime('%m/%d/%Y')}"
        )
    else:
        post_peak_text = "N/A"
    # --------------------------------------------------------
    # END CUMULATIVE
    # --------------------------------------------------------
    end_cumulative = float(
        d.iloc[-1][
            cum_col
        ]
    )
    # --------------------------------------------------------
    # BUILD FIGURE
    # --------------------------------------------------------
    fig, ax1 = plt.subplots(
        figsize=(14, 7)
    )
    ax2 = ax1.twinx()
    x = list(
        d["Date"].dt.to_pydatetime()
    )
    current_rate = d[
        rate_col
    ].to_numpy(dtype=float)
    current_cum = d[
        cum_col
    ].to_numpy(dtype=float)
    # ========================================================
    # BASE CASE
    # ========================================================
    if prior_df is None:
        line_rate, = ax1.plot(
            x,
            current_rate,
            linewidth=2,
            label=f"{case_label} Rate"
        )
        line_cum, = ax2.plot(
            x,
            current_cum,
            linestyle="--",
            linewidth=2,
            label=f"{case_label} Cumulative"
        )
        handles = [
            line_rate,
            line_cum
        ]
        labels = [
            f"{case_label} Rate",
            f"{case_label} Cumulative"
        ]
    # ========================================================
    # DEVELOPMENT WEDGE CASE
    # ========================================================
    else:
        p = prior_df[
            prior_df["Facility"] == facility
        ].copy()
        p = p[
            p["Date"] <= pdp_end
        ].copy()
        p = p[
            [
                "Date",
                rate_col,
                cum_col
            ]
        ].rename(
            columns={
                rate_col:
                    "Prior_Rate",
                cum_col:
                    "Prior_Cum"
            }
        )
        joined = d.merge(
            p,
            on="Date",
            how="left"
        )
        joined[
            [
                "Prior_Rate",
                "Prior_Cum"
            ]
        ] = joined[
            [
                "Prior_Rate",
                "Prior_Cum"
            ]
        ].fillna(0.0)
        x = list(
            joined[
                "Date"
            ].dt.to_pydatetime()
        )
        current_rate = joined[
            rate_col
        ].to_numpy(dtype=float)
        current_cum = joined[
            cum_col
        ].to_numpy(dtype=float)
        prior_rate = joined[
            "Prior_Rate"
        ].to_numpy(dtype=float)
        prior_cum = joined[
            "Prior_Cum"
        ].to_numpy(dtype=float)
        line_prior_rate, = ax1.plot(
            x,
            prior_rate,
            linewidth=1.6,
            label=f"{prior_label} Rate"
        )
        line_current_rate, = ax1.plot(
            x,
            current_rate,
            linewidth=2,
            label=f"{case_label} Rate"
        )
        # Development rate wedge
        ax1.fill_between(
            x,
            prior_rate,
            current_rate,
            alpha=0.22
        )
        line_prior_cum, = ax2.plot(
            x,
            prior_cum,
            linestyle="--",
            linewidth=1.6,
            label=f"{prior_label} Cumulative"
        )
        line_current_cum, = ax2.plot(
            x,
            current_cum,
            linestyle="--",
            linewidth=2,
            label=f"{case_label} Cumulative"
        )
        handles = [
            line_prior_rate,
            line_current_rate,
            line_prior_cum,
            line_current_cum
        ]
        labels = [
            f"{prior_label} Rate",
            f"{case_label} Rate",
            f"{prior_label} Cumulative",
            f"{case_label} Cumulative"
        ]
    # --------------------------------------------------------
    # ANNOTATION
    # --------------------------------------------------------
    annotation = (
        f"Peak Rate: "
        f"{peak_rate:,.0f} {rate_unit} "
        f"on {peak_date.strftime('%m/%d/%Y')}\n"
        f"Post-2034 Peak: "
        f"{post_peak_text}\n"
        f"Rate on 2/1/2034: "
        f"{rate_check_text}\n"
        f"End Cumulative: "
        f"{end_cumulative:,.0f} "
        f"{cum_unit}"
    )
    ax1.text(
        0.015,
        0.97,
        annotation,
        transform=ax1.transAxes,
        va="top",
        ha="left",
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.88
        )
    )
    ax1.set_title(
        f"{facility} - "
        f"{stream} - "
        f"{case_label}"
    )
    ax1.set_xlabel(
        "Date"
    )
    ax1.set_ylabel(
        f"{stream} Rate ({rate_unit})"
    )
    ax2.set_ylabel(
        f"Cumulative {stream} ({cum_unit})"
    )
    ax1.grid(
        True,
        alpha=0.30
    )
    ax1.legend(
        handles,
        labels,
        loc="best"
    )
    plt.tight_layout()
    plt.show()
# ============================================================
# CASE 1
# ALL PDP BASE FACILITIES
# ============================================================
for facility in sorted(
    case1_plot["Facility"]
    .dropna()
    .unique()
):
    plot_profile_final(
        case1_plot,
        facility,
        "PDP Base",
        "Oil"
    )
    plot_profile_final(
        case1_plot,
        facility,
        "PDP Base",
        "Gas"
    )
# ============================================================
# CASE 2
# ONLY FACILITIES WITH CONFIRMED DEVELOPMENT
# ============================================================
for facility in sorted(
    confirmed_affected
):
    plot_profile_final(
        case2_plot,
        facility,
        "PDP + Confirmed Dev",
        "Oil",
        prior_df=case1_plot,
        prior_label="PDP Base"
    )
    plot_profile_final(
        case2_plot,
        facility,
        "PDP + Confirmed Dev",
        "Gas",
        prior_df=case1_plot,
        prior_label="PDP Base"
    )
# ============================================================
# CASE 3
# ONLY FACILITIES WITH UNCERTAIN DEVELOPMENT
# ============================================================
for facility in sorted(
    uncertain_affected
):
    plot_profile_final(
        case3_plot,
        facility,
        "PDP + Confirmed + Uncertain Dev",
        "Oil",
        prior_df=case2_plot,
        prior_label="PDP + Confirmed Dev"
    )
    plot_profile_final(
        case3_plot,
        facility,
        "PDP + Confirmed + Uncertain Dev",
        "Gas",
        prior_df=case2_plot,
        prior_label="PDP + Confirmed Dev"
    )
print("Finished regenerating all profiles.")



In [ ]:
# ============================================================
# FACILITY RATE SENSITIVITY SUMMARY + RANKINGS
#
# Outputs, for every facility and every case:
#
# 1) Oil & Gas rate on 2/1/2034
# 2) Peak Oil & Gas rate AFTER/ON 2/1/2034 + peak date
# 3) Current Oil & Gas rate
#
# Rankings are calculated using ONLY the most aggressive case:
# PDP + Confirmed + Uncertain Dev
#
# Rank 1 = highest rate
#
# Saves:
# Facility_Rate_Sensitivity_Summary.csv
# ============================================================
import pandas as pd
import numpy as np
# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------
TARGET_DATE = pd.Timestamp("2034-02-01")
# Current month, normalized to first of month
CURRENT_MONTH = (
    pd.Timestamp.today()
    .to_period("M")
    .to_timestamp()
)
OUTPUT_FILE = "Facility_Rate_Sensitivity_Summary.csv"
# ------------------------------------------------------------
# PREPARE EACH CASE
# ------------------------------------------------------------
def prepare_rate_case(df):
    d = df.copy()
    d["Date"] = pd.to_datetime(
        d["Date"],
        errors="coerce"
    )
    d = d[
        d["Date"].notna()
    ].copy()
    d = d.sort_values(
        ["Facility", "Date"]
    ).reset_index(drop=True)
    # Make sure total monthly volumes are numeric
    d["Total_Oil_BBL"] = pd.to_numeric(
        d["Total_Oil_BBL"],
        errors="coerce"
    ).fillna(0.0)
    d["Total_Gas_MCF"] = pd.to_numeric(
        d["Total_Gas_MCF"],
        errors="coerce"
    ).fillna(0.0)
    # Convert monthly volumes to average calendar-day rates
    d["Days_In_Month"] = (
        d["Date"].dt.days_in_month
    )
    d["Oil_BPD"] = (
        d["Total_Oil_BBL"]
        /
        d["Days_In_Month"]
    )
    d["Gas_MCFD"] = (
        d["Total_Gas_MCF"]
        /
        d["Days_In_Month"]
    )
    return d
case1_rates = prepare_rate_case(case1)
case2_rates = prepare_rate_case(case2)
case3_rates = prepare_rate_case(case3)
# ------------------------------------------------------------
# BUILD ONE SUMMARY ROW PER FACILITY / CASE
# ------------------------------------------------------------
def summarize_case_rates(df, case_name):
    rows = []
    for facility, d in df.groupby("Facility"):
        d = (
            d.sort_values("Date")
            .copy()
        )
        # ====================================================
        # CURRENT RATE
        # Most recent month available on/before current month
        # ====================================================
        current_rows = d[
            d["Date"] <= CURRENT_MONTH
        ]
        if len(current_rows) > 0:
            current_row = (
                current_rows
                .iloc[-1]
            )
            current_date = (
                current_row["Date"]
            )
            current_oil = float(
                current_row["Oil_BPD"]
            )
            current_gas = float(
                current_row["Gas_MCFD"]
            )
        else:
            current_date = pd.NaT
            current_oil = np.nan
            current_gas = np.nan
        # ====================================================
        # RATE ON 2/1/2034
        # ====================================================
        target_row = d[
            d["Date"] == TARGET_DATE
        ]
        if len(target_row) > 0:
            oil_2034 = float(
                target_row.iloc[0][
                    "Oil_BPD"
                ]
            )
            gas_2034 = float(
                target_row.iloc[0][
                    "Gas_MCFD"
                ]
            )
        else:
            oil_2034 = np.nan
            gas_2034 = np.nan
        # ====================================================
        # PEAK RATE ON/AFTER 2/1/2034
        # ====================================================
        post = d[
            d["Date"] >= TARGET_DATE
        ].copy()
        if len(post) > 0:
            # Oil
            oil_idx = (
                post["Oil_BPD"]
                .idxmax()
            )
            post_peak_oil = float(
                post.loc[
                    oil_idx,
                    "Oil_BPD"
                ]
            )
            post_peak_oil_date = (
                post.loc[
                    oil_idx,
                    "Date"
                ]
            )
            # Gas
            gas_idx = (
                post["Gas_MCFD"]
                .idxmax()
            )
            post_peak_gas = float(
                post.loc[
                    gas_idx,
                    "Gas_MCFD"
                ]
            )
            post_peak_gas_date = (
                post.loc[
                    gas_idx,
                    "Date"
                ]
            )
        else:
            post_peak_oil = np.nan
            post_peak_oil_date = pd.NaT
            post_peak_gas = np.nan
            post_peak_gas_date = pd.NaT
        # ====================================================
        # STORE ROW
        # ====================================================
        rows.append({
            "Case":
                case_name,
            "Facility":
                facility,
            "Current_Rate_Date":
                current_date,
            "Current_Oil_BPD":
                current_oil,
            "Current_Gas_MCFD":
                current_gas,
            "Oil_BPD_2_1_2034":
                oil_2034,
            "Gas_MCFD_2_1_2034":
                gas_2034,
            "Post_2_1_2034_Peak_Oil_BPD":
                post_peak_oil,
            "Post_2_1_2034_Peak_Oil_Date":
                post_peak_oil_date,
            "Post_2_1_2034_Peak_Gas_MCFD":
                post_peak_gas,
            "Post_2_1_2034_Peak_Gas_Date":
                post_peak_gas_date
        })
    return pd.DataFrame(rows)
# ------------------------------------------------------------
# SUMMARIZE ALL THREE CASES
# ------------------------------------------------------------
summary1 = summarize_case_rates(
    case1_rates,
    "PDP Base"
)
summary2 = summarize_case_rates(
    case2_rates,
    "PDP + Confirmed Dev"
)
summary3 = summarize_case_rates(
    case3_rates,
    "PDP + Confirmed + Uncertain Dev"
)
all_summary = pd.concat(
    [
        summary1,
        summary2,
        summary3
    ],
    ignore_index=True
)
# ============================================================
# RANK FACILITIES USING MOST AGGRESSIVE CASE ONLY
#
# 1 = highest rate
#
# Six separate rankings:
#
# CURRENT
#   Oil
#   Gas
#
# 2/1/2034
#   Oil
#   Gas
#
# POST-2/1/2034 PEAK
#   Oil
#   Gas
# ============================================================
aggressive = summary3.copy()
# Current rankings
aggressive["Aggressive_Current_Oil_Rank"] = (
    aggressive["Current_Oil_BPD"]
    .rank(
        method="min",
        ascending=False
    )
)
aggressive["Aggressive_Current_Gas_Rank"] = (
    aggressive["Current_Gas_MCFD"]
    .rank(
        method="min",
        ascending=False
    )
)
# 2/1/2034 rankings
aggressive["Aggressive_2_1_2034_Oil_Rank"] = (
    aggressive["Oil_BPD_2_1_2034"]
    .rank(
        method="min",
        ascending=False
    )
)
aggressive["Aggressive_2_1_2034_Gas_Rank"] = (
    aggressive["Gas_MCFD_2_1_2034"]
    .rank(
        method="min",
        ascending=False
    )
)
# Post-2034 peak rankings
aggressive["Aggressive_Post2034_Peak_Oil_Rank"] = (
    aggressive[
        "Post_2_1_2034_Peak_Oil_BPD"
    ]
    .rank(
        method="min",
        ascending=False
    )
)
aggressive["Aggressive_Post2034_Peak_Gas_Rank"] = (
    aggressive[
        "Post_2_1_2034_Peak_Gas_MCFD"
    ]
    .rank(
        method="min",
        ascending=False
    )
)
# Convert rankings to nullable integers so they display cleanly
rank_columns = [
    "Aggressive_Current_Oil_Rank",
    "Aggressive_Current_Gas_Rank",
    "Aggressive_2_1_2034_Oil_Rank",
    "Aggressive_2_1_2034_Gas_Rank",
    "Aggressive_Post2034_Peak_Oil_Rank",
    "Aggressive_Post2034_Peak_Gas_Rank"
]
for col in rank_columns:
    aggressive[col] = (
        aggressive[col]
        .astype("Int64")
    )
# ------------------------------------------------------------
# MERGE AGGRESSIVE-CASE FACILITY RANKS ONTO EVERY CASE ROW
#
# This makes it easy to see the same facility ranking next
# to Base, CFD, and UFD sensitivity outputs.
# ------------------------------------------------------------
rank_table = aggressive[
    [
        "Facility"
    ]
    +
    rank_columns
].copy()
all_summary = all_summary.merge(
    rank_table,
    on="Facility",
    how="left"
)
# ------------------------------------------------------------
# COLUMN ORDER
# ------------------------------------------------------------
all_summary = all_summary[
    [
        "Facility",
        "Case",
        "Current_Rate_Date",
        "Current_Oil_BPD",
        "Aggressive_Current_Oil_Rank",
        "Current_Gas_MCFD",
        "Aggressive_Current_Gas_Rank",
        "Oil_BPD_2_1_2034",
        "Aggressive_2_1_2034_Oil_Rank",
        "Gas_MCFD_2_1_2034",
        "Aggressive_2_1_2034_Gas_Rank",
        "Post_2_1_2034_Peak_Oil_BPD",
        "Post_2_1_2034_Peak_Oil_Date",
        "Aggressive_Post2034_Peak_Oil_Rank",
        "Post_2_1_2034_Peak_Gas_MCFD",
        "Post_2_1_2034_Peak_Gas_Date",
        "Aggressive_Post2034_Peak_Gas_Rank"
    ]
]
# ------------------------------------------------------------
# SORT
# Facility together, cases in logical order
# ------------------------------------------------------------
case_order = {
    "PDP Base": 1,
    "PDP + Confirmed Dev": 2,
    "PDP + Confirmed + Uncertain Dev": 3
}
all_summary["_Case_Order"] = (
    all_summary["Case"]
    .map(case_order)
)
all_summary = (
    all_summary
    .sort_values(
        [
            "Facility",
            "_Case_Order"
        ]
    )
    .drop(
        columns="_Case_Order"
    )
    .reset_index(drop=True)
)
# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------
all_summary.to_csv(
    OUTPUT_FILE,
    index=False
)
print(
    f"Saved: {OUTPUT_FILE}"
)
print(
    "Current rate month used:",
    CURRENT_MONTH.strftime("%m/%d/%Y")
)
print(
    "Rank 1 = highest rate in "
    "PDP + Confirmed + Uncertain Dev case."
)
display(all_summary)


In [ ]:
# =====================================================================
# PDP RETURN-TO-PRODUCTION (RTP) SENSITIVITY
#
# HIGH / ON
# ---------------------------------------------------------------------
# Normal PDP:
#     Actual production through last actual month
#     -> massive PDP forecast thereafter
#
# Sensitivity Wells:
#     Actual production through last actual month
#     -> massive PDP forecast thereafter
#
# Problematic DSUs:
#     COMPLETELY REPLACE BOTH ORIGINAL PRODUCTION AND FORECAST
#     with the alternative type curve.
#
#     TC Forecast Month 1 is placed at the well's FIRST ORIGINAL
#     PRODUCTION MONTH.
#
#     If the well has no actual production, Forecast Month 1 is
#     placed at the first available massive PDP forecast month.
#
#
# LOW / OFF
# ---------------------------------------------------------------------
# Normal PDP:
#     Actual -> massive PDP forecast
#
# Sensitivity Wells:
#     Actual -> ZERO after last actual
#
# Problematic DSUs:
#     Actual -> massive PDP forecast
#
#
# CASES
# ---------------------------------------------------------------------
# 01 = PDP
# 02 = PDP + CFD
# 03 = PDP + CFD + UFD
#
#
# HERCULES SPECIAL HANDLING
# ---------------------------------------------------------------------
# OIL:
#   HERCULES CPF
#   HERCULES EAST SATELLITE
#   HERCULES WEST SATELLITE
#
#       -> combined into one "HERCULES" oil facility
#
# GAS:
#   HERCULES CPF
#   HERCULES EAST SATELLITE
#   HERCULES WEST SATELLITE
#
#       -> remain three separate gas facilities
#
#
# CURRENT-RATE REPORTING
# ---------------------------------------------------------------------
# Forecast_Dependent_Current_Rate
#     = 8/1/2026 modeled/stiched profile
#
# Recorded_PDP_Rate_7_15_2026
#     = actual recorded July PDP production
#     = directly from the PDP production source
#
# NOTE:
# The July recorded-production metric remains the REAL recorded value
# even for Problematic DSUs. It is a reference metric, not the modeled
# HIGH case.
#
#
# SUMMARY
# ---------------------------------------------------------------------
# Includes:
#   - recorded PDP actual rate (July 2026)
#   - forecast-dependent current rate (August 2026)
#   - 2/1/2034 rate
#   - peak rate on/after 2/1/2034 + peak date
#   - ending cumulative
#   - HIGH minus LOW deltas
#   - Case 03 facility rankings
#
# Rank 1 = highest rate
# =====================================================================
import pandas as pd
import numpy as np
import re
# =====================================================================
# FILES / SETTINGS
# =====================================================================
SENSITIVITY_WELLS_FILE = "Sensitivity Wells.csv"
PROBLEMATIC_DSUS_FILE = "Problematic DSUs.csv"
# Change only this filename if your TC CSV has a different name.
PROBLEMATIC_DSU_TC_FILE = "MONTHLY FRCST CMG SENS 6 COL.csv"
# Existing "current" modeled-rate convention
FORECAST_CURRENT_DATE = pd.Timestamp("2026-08-01")
# Actual recorded-production reference
RECORDED_PRODUCTION_DATE = pd.Timestamp("2026-07-01")
# Future reporting date
TARGET_2034_DATE = pd.Timestamp("2034-02-01")
SUMMARY_FILE = (
    "PDP_RTP_Sensitivity_Facility_Summary.csv"
)
# =====================================================================
# FACILITY NORMALIZATION
# =====================================================================
def rtp_normalize_facility(v):
    if pd.isna(v):
        return ""
    return re.sub(
        r"\s+",
        " ",
        str(v).strip()
    ).upper()
# =====================================================================
# HERCULES FACILITY HANDLING
# =====================================================================
HERCULES_NAMES = {
    "HERCULES CPF",
    "HERCULES EAST SATELLITE",
    "HERCULES WEST SATELLITE"
}
def oil_facility_map(v):
    f = rtp_normalize_facility(v)
    if f in HERCULES_NAMES:
        return "HERCULES"
    return f
def gas_facility_map(v):
    # Gas intentionally retains the satellite distinction.
    return rtp_normalize_facility(v)
# =====================================================================
# READ SIMPLE COLUMN-A WELL LISTS
# =====================================================================
def read_simple_well_list(filename):
    d = pd.read_csv(
        filename,
        usecols=[0]
    )
    d.columns = ["Well_Name"]
    d["Match_Name"] = (
        d["Well_Name"]
        .apply(normalize_name)
    )
    return set(
        d.loc[
            d["Match_Name"] != "",
            "Match_Name"
        ]
    )
SENSITIVITY_WELLS = read_simple_well_list(
    SENSITIVITY_WELLS_FILE
)
PROBLEMATIC_DSUS = read_simple_well_list(
    PROBLEMATIC_DSUS_FILE
)
print(
    "Sensitivity Wells:",
    len(SENSITIVITY_WELLS)
)
print(
    "Problematic DSUs:",
    len(PROBLEMATIC_DSUS)
)
# =====================================================================
# DO NOT ALLOW A WELL IN BOTH LISTS
# =====================================================================
LIST_OVERLAP = (
    SENSITIVITY_WELLS
    &
    PROBLEMATIC_DSUS
)
if len(LIST_OVERLAP) > 0:
    print()
    print(
        "ERROR: These wells occur in BOTH "
        "Sensitivity Wells and Problematic DSUs:"
    )
    for w in sorted(LIST_OVERLAP):
        print(w)
    raise ValueError(
        "Remove overlapping wells from one of the two lists."
    )
# =====================================================================
# PREP PDP ROUTING
# =====================================================================
pdp_route_rtp = pdp_route.copy()
pdp_route_rtp["Facility"] = (
    pdp_route_rtp["Facility"]
    .apply(rtp_normalize_facility)
)
pdp_route_rtp["Match_Name"] = (
    pdp_route_rtp["Match_Name"]
    .apply(normalize_name)
)
pdp_route_rtp["WI"] = pd.to_numeric(
    pdp_route_rtp["WI"],
    errors="coerce"
)
if pdp_route_rtp["WI"].isna().any():
    raise ValueError(
        "PDP routing contains blank/non-numeric WI."
    )
# Protect against WI being stored as 0-100 rather than decimal.
if (
    len(pdp_route_rtp) > 0
    and
    pdp_route_rtp["WI"].max() > 1.0
):
    pdp_route_rtp["WI"] = (
        pdp_route_rtp["WI"]
        /
        100.0
    )
PDP_ALLOWED_RTP = set(
    pdp_route_rtp["Match_Name"]
)
# =====================================================================
# CHECK LISTED WELLS AGAINST PDP ROUTING
# =====================================================================
missing_sensitivity = sorted(
    SENSITIVITY_WELLS
    -
    PDP_ALLOWED_RTP
)
missing_problematic = sorted(
    PROBLEMATIC_DSUS
    -
    PDP_ALLOWED_RTP
)
if len(missing_sensitivity) > 0:
    print()
    print(
        "WARNING - Sensitivity Wells not found in PDP routing:"
    )
    for w in missing_sensitivity:
        print(w)
if len(missing_problematic) > 0:
    print()
    print(
        "WARNING - Problematic DSUs not found in PDP routing:"
    )
    for w in missing_problematic:
        print(w)
# =====================================================================
# READ MASSIVE PDP PRODUCTION / FORECAST FILE
# =====================================================================
def read_large_pdp_rtp(filename):
    keep = []
    for chunk in pd.read_csv(
        filename,
        usecols=[0, 7, 8, 9],
        chunksize=CHUNK_SIZE
    ):
        chunk.columns = [
            "Well_Name",
            "Date",
            "Oil_BBL",
            "Gas_MCF"
        ]
        chunk["Match_Name"] = (
            chunk["Well_Name"]
            .apply(normalize_name)
        )
        chunk = chunk[
            chunk["Match_Name"].isin(
                PDP_ALLOWED_RTP
            )
        ].copy()
        if len(chunk) == 0:
            continue
        # Normalize 7/15/26 -> 7/1/26, 8/15/26 -> 8/1/26, etc.
        chunk["Date"] = (
            pd.to_datetime(
                chunk["Date"],
                errors="coerce"
            )
            .dt.to_period("M")
            .dt.to_timestamp()
        )
        chunk["Oil_BBL"] = pd.to_numeric(
            chunk["Oil_BBL"],
            errors="coerce"
        ).fillna(0.0)
        chunk["Gas_MCF"] = pd.to_numeric(
            chunk["Gas_MCF"],
            errors="coerce"
        ).fillna(0.0)
        chunk = chunk[
            chunk["Date"].notna()
        ].copy()
        keep.append(
            chunk.groupby(
                [
                    "Match_Name",
                    "Date"
                ],
                as_index=False
            )[
                [
                    "Oil_BBL",
                    "Gas_MCF"
                ]
            ].sum()
        )
    if len(keep) == 0:
        return pd.DataFrame(
            columns=[
                "Match_Name",
                "Date",
                "Oil_BBL",
                "Gas_MCF"
            ]
        )
    d = pd.concat(
        keep,
        ignore_index=True
    )
    return (
        d.groupby(
            [
                "Match_Name",
                "Date"
            ],
            as_index=False
        )[
            [
                "Oil_BBL",
                "Gas_MCF"
            ]
        ]
        .sum()
    )
print()
print("Reading PDP actual production...")
pdp_actual_rtp = read_large_pdp_rtp(
    PDP_PRODUCTION_FILE
)
print("Reading massive PDP forecast...")
pdp_forecast_rtp = read_large_pdp_rtp(
    PDP_FORECAST_FILE
)
# =====================================================================
# READ PROBLEMATIC DSU TYPE-CURVE FILE
#
# Expected file format:
#
# A = ENTITY_NAME
# B = FORECAST MONTHS
# C = OIL VOLUME
# D = OIL RATE
# E = RAW GAS VOLUME
# F = RAW GAS RATE
#
# We use:
#   A = well
#   B = forecast month
#   C = monthly oil volume
#   E = monthly gas volume
#
# Rate columns are NOT used because rate is recalculated from
# monthly volume / calendar days.
# =====================================================================
problem_tc = pd.read_csv(
    PROBLEMATIC_DSU_TC_FILE,
    usecols=[0, 1, 2, 4]
)
problem_tc.columns = [
    "Well_Name",
    "Forecast_Month",
    "Oil_BBL",
    "Gas_MCF"
]
problem_tc["Match_Name"] = (
    problem_tc["Well_Name"]
    .apply(normalize_name)
)
problem_tc["Forecast_Month"] = pd.to_numeric(
    problem_tc["Forecast_Month"],
    errors="coerce"
)
problem_tc["Oil_BBL"] = pd.to_numeric(
    problem_tc["Oil_BBL"],
    errors="coerce"
).fillna(0.0)
problem_tc["Gas_MCF"] = pd.to_numeric(
    problem_tc["Gas_MCF"],
    errors="coerce"
).fillna(0.0)
problem_tc = problem_tc[
    problem_tc["Forecast_Month"].notna()
].copy()
problem_tc["Forecast_Month"] = (
    problem_tc["Forecast_Month"]
    .astype(int)
)
problem_tc = problem_tc[
    problem_tc["Match_Name"].isin(
        PROBLEMATIC_DSUS
    )
].copy()
# =====================================================================
# CHECK TYPE-CURVE COVERAGE
# =====================================================================
tc_names = set(
    problem_tc["Match_Name"]
)
missing_tc = sorted(
    PROBLEMATIC_DSUS
    -
    tc_names
)
if len(missing_tc) > 0:
    print()
    print(
        "ERROR - Problematic DSUs missing from "
        "Problematic DSU Type Curve file:"
    )
    for w in missing_tc:
        print(w)
    raise ValueError(
        "Every Problematic DSU must have a type curve."
    )
print()
print(
    "Problematic DSUs with alternative type curves:",
    len(tc_names)
)
# =====================================================================
# GET FIRST ORIGINAL ONLINE MONTH FOR A PROBLEMATIC DSU
#
# HIGH case TC Forecast Month 1 begins here.
#
# Priority:
#   1. first actual production month
#   2. if no actual exists, first massive PDP forecast month
#
# This is intentionally NOT the month after last actual.
#
# Because HIGH completely replaces BOTH actual production AND forecast.
# =====================================================================
def get_problematic_tc_start(
    well,
    p,
    f
):
    if len(p) > 0:
        return p[
            "Date"
        ].min()
    if len(f) > 0:
        return f[
            "Date"
        ].min()
    raise ValueError(
        f"{well}: no production or forecast date exists "
        "to anchor the Problematic DSU type curve."
    )
# =====================================================================
# CREATE DATE-ALIGNED PROBLEMATIC DSU TYPE CURVE
#
# Example:
#
# Original first production = 4/15/2026
# Normalized month          = 4/1/2026
#
# TC Forecast Month 1       = 4/1/2026
# TC Forecast Month 2       = 5/1/2026
# etc.
# =====================================================================
def get_problematic_tc_profile(
    well,
    tc_start
):
    tc = problem_tc[
        problem_tc["Match_Name"]
        ==
        well
    ].copy()
    if len(tc) == 0:
        raise ValueError(
            f"{well}: no Problematic DSU type curve found."
        )
    tc = tc.sort_values(
        "Forecast_Month"
    ).copy()
    tc["Date"] = [
        pd.Timestamp(tc_start)
        +
        pd.DateOffset(
            months=int(m) - 1
        )
        for m in tc[
            "Forecast_Month"
        ]
    ]
    tc = tc.rename(
        columns={
            "Oil_BBL":
                "TC_Oil_BBL",
            "Gas_MCF":
                "TC_Gas_MCF"
        }
    )
    return tc[
        [
            "Date",
            "TC_Oil_BBL",
            "TC_Gas_MCF"
        ]
    ]
# =====================================================================
# BUILD HIGH / LOW PDP WELL PROFILES
# =====================================================================
def build_rtp_pdp_profiles(
    scenario
):
    if scenario not in [
        "HIGH_ON",
        "LOW_OFF"
    ]:
        raise ValueError(
            "scenario must be HIGH_ON or LOW_OFF"
        )
    pieces = []
    for _, route in pdp_route_rtp.iterrows():
        well = route[
            "Match_Name"
        ]
        facility = route[
            "Facility"
        ]
        wi = float(
            route["WI"]
        )
        # -------------------------------------------------------------
        # RAW ORIGINAL ACTUAL PRODUCTION
        # -------------------------------------------------------------
        p = pdp_actual_rtp[
            pdp_actual_rtp[
                "Match_Name"
            ]
            ==
            well
        ].copy()
        # -------------------------------------------------------------
        # RAW ORIGINAL MASSIVE PDP FORECAST
        # -------------------------------------------------------------
        f = pdp_forecast_rtp[
            pdp_forecast_rtp[
                "Match_Name"
            ]
            ==
            well
        ].copy()
        # =============================================================
        # PROBLEMATIC DSU + HIGH / ON
        #
        # COMPLETELY IGNORE BOTH p AND f FOR VOLUME.
        #
        # They are used ONLY to determine the original first online
        # month to which TC Forecast Month 1 is anchored.
        # =============================================================
        if (
            well in PROBLEMATIC_DSUS
            and
            scenario == "HIGH_ON"
        ):
            tc_start = get_problematic_tc_start(
                well,
                p,
                f
            )
            tc = get_problematic_tc_profile(
                well,
                tc_start
            )
            # Restrict to requested maximum well life.
            tc = (
                tc.sort_values(
                    "Date"
                )
                .head(
                    MAX_WELL_MONTHS
                )
                .copy()
            )
            high_problem = pd.DataFrame(
                {
                    "Facility":
                        facility,
                    "Match_Name":
                        well,
                    "Date":
                        tc["Date"],
                    # TC completely replaces actual + PDP forecast.
                    "Net_Oil_BBL":
                        tc[
                            "TC_Oil_BBL"
                        ].to_numpy(dtype=float)
                        *
                        wi,
                    "Net_Gas_MCF":
                        tc[
                            "TC_Gas_MCF"
                        ].to_numpy(dtype=float)
                        *
                        wi
                }
            )
            pieces.append(
                high_problem
            )
            # Critical:
            # do NOT run normal PDP stitching for this well.
            continue
        # =============================================================
        # ALL OTHER SITUATIONS
        #
        # This includes:
        #
        # LOW/OFF Problematic DSU
        # HIGH/ON Sensitivity Well
        # LOW/OFF Sensitivity Well
        # Normal PDP in either scenario
        # =============================================================
        # -------------------------------------------------------------
        # PROFILE START
        # -------------------------------------------------------------
        if len(p) > 0:
            start = p[
                "Date"
            ].min()
        elif len(f) > 0:
            start = f[
                "Date"
            ].min()
        else:
            continue
        calendar = pd.DataFrame(
            {
                "Date":
                    pd.date_range(
                        start=start,
                        periods=MAX_WELL_MONTHS,
                        freq="MS"
                    )
            }
        )
        # -------------------------------------------------------------
        # ORIGINAL ACTUAL PRODUCTION
        # -------------------------------------------------------------
        actual = p[
            [
                "Date",
                "Oil_BBL",
                "Gas_MCF"
            ]
        ].rename(
            columns={
                "Oil_BBL":
                    "Actual_Oil_BBL",
                "Gas_MCF":
                    "Actual_Gas_MCF"
            }
        )
        calendar = calendar.merge(
            actual,
            on="Date",
            how="left"
        )
        # -------------------------------------------------------------
        # ORIGINAL MASSIVE PDP FORECAST
        # -------------------------------------------------------------
        forecast = f[
            [
                "Date",
                "Oil_BBL",
                "Gas_MCF"
            ]
        ].rename(
            columns={
                "Oil_BBL":
                    "Forecast_Oil_BBL",
                "Gas_MCF":
                    "Forecast_Gas_MCF"
            }
        )
        calendar = calendar.merge(
            forecast,
            on="Date",
            how="left"
        )
        for col in [
            "Actual_Oil_BBL",
            "Actual_Gas_MCF",
            "Forecast_Oil_BBL",
            "Forecast_Gas_MCF"
        ]:
            if col not in calendar.columns:
                calendar[col] = 0.0
            calendar[col] = (
                pd.to_numeric(
                    calendar[col],
                    errors="coerce"
                )
                .fillna(0.0)
            )
        # -------------------------------------------------------------
        # LAST ACTUAL PRODUCTION
        # -------------------------------------------------------------
        if len(p) > 0:
            last_actual = (
                p["Date"].max()
            )
            actual_mask = (
                calendar["Date"]
                <=
                last_actual
            )
        else:
            last_actual = pd.NaT
            actual_mask = pd.Series(
                False,
                index=calendar.index
            )
        future_mask = (
            ~actual_mask
        )
        # -------------------------------------------------------------
        # START PROFILE AT ZERO
        # -------------------------------------------------------------
        calendar[
            "Gross_Oil_BBL"
        ] = 0.0
        calendar[
            "Gross_Gas_MCF"
        ] = 0.0
        # -------------------------------------------------------------
        # ORIGINAL ACTUAL PRODUCTION
        # -------------------------------------------------------------
        calendar.loc[
            actual_mask,
            "Gross_Oil_BBL"
        ] = calendar.loc[
            actual_mask,
            "Actual_Oil_BBL"
        ]
        calendar.loc[
            actual_mask,
            "Gross_Gas_MCF"
        ] = calendar.loc[
            actual_mask,
            "Actual_Gas_MCF"
        ]
        # =============================================================
        # FUTURE FORECAST LOGIC
        # =============================================================
        # -------------------------------------------------------------
        # SENSITIVITY WELL
        # -------------------------------------------------------------
        if well in SENSITIVITY_WELLS:
            if scenario == "HIGH_ON":
                # HIGH / ON:
                # retain massive PDP forecast
                calendar.loc[
                    future_mask,
                    "Gross_Oil_BBL"
                ] = calendar.loc[
                    future_mask,
                    "Forecast_Oil_BBL"
                ]
                calendar.loc[
                    future_mask,
                    "Gross_Gas_MCF"
                ] = calendar.loc[
                    future_mask,
                    "Forecast_Gas_MCF"
                ]
            else:
                # LOW / OFF:
                # ALL forecast after last actual remains ZERO.
                pass
        # -------------------------------------------------------------
        # PROBLEMATIC DSU IN LOW / OFF
        # -------------------------------------------------------------
        elif well in PROBLEMATIC_DSUS:
            # HIGH already exited above.
            #
            # Therefore this section is LOW / OFF only:
            # actual production + massive PDP forecast.
            calendar.loc[
                future_mask,
                "Gross_Oil_BBL"
            ] = calendar.loc[
                future_mask,
                "Forecast_Oil_BBL"
            ]
            calendar.loc[
                future_mask,
                "Gross_Gas_MCF"
            ] = calendar.loc[
                future_mask,
                "Forecast_Gas_MCF"
            ]
        # -------------------------------------------------------------
        # NORMAL PDP
        # -------------------------------------------------------------
        else:
            calendar.loc[
                future_mask,
                "Gross_Oil_BBL"
            ] = calendar.loc[
                future_mask,
                "Forecast_Oil_BBL"
            ]
            calendar.loc[
                future_mask,
                "Gross_Gas_MCF"
            ] = calendar.loc[
                future_mask,
                "Forecast_Gas_MCF"
            ]
        # -------------------------------------------------------------
        # APPLY WI
        # -------------------------------------------------------------
        calendar[
            "Net_Oil_BBL"
        ] = (
            calendar[
                "Gross_Oil_BBL"
            ]
            *
            wi
        )
        calendar[
            "Net_Gas_MCF"
        ] = (
            calendar[
                "Gross_Gas_MCF"
            ]
            *
            wi
        )
        calendar[
            "Facility"
        ] = facility
        calendar[
            "Match_Name"
        ] = well
        pieces.append(
            calendar[
                [
                    "Facility",
                    "Match_Name",
                    "Date",
                    "Net_Oil_BBL",
                    "Net_Gas_MCF"
                ]
            ]
        )
    if len(pieces) == 0:
        return pd.DataFrame(
            columns=[
                "Facility",
                "Match_Name",
                "Date",
                "Net_Oil_BBL",
                "Net_Gas_MCF"
            ]
        )
    return pd.concat(
        pieces,
        ignore_index=True
    )
# =====================================================================
# BUILD HIGH + LOW PDP PROFILES
# =====================================================================
print()
print(
    "Building HIGH / ON PDP profile..."
)
pdp_high_wells = build_rtp_pdp_profiles(
    "HIGH_ON"
)
print(
    "Building LOW / OFF PDP profile..."
)
pdp_low_wells = build_rtp_pdp_profiles(
    "LOW_OFF"
)
# =====================================================================
# DIAGNOSTIC CHECK
#
# Confirms that HIGH Problematic DSUs begin with TC Forecast Month 1
# and are not stitched to actual production.
# =====================================================================
print()
print(
    "Problematic DSU HIGH-case TC start check:"
)
for well in sorted(
    PROBLEMATIC_DSUS
):
    x = pdp_high_wells[
        pdp_high_wells[
            "Match_Name"
        ]
        ==
        well
    ].sort_values(
        "Date"
    )
    if len(x) > 0:
        print(
            well,
            "-> HIGH profile starts",
            x.iloc[0]["Date"].strftime(
                "%m/%d/%Y"
            )
        )
# =====================================================================
# AGGREGATE PDP BY FACILITY
#
# OIL:
#   all Hercules components combined
#
# GAS:
#   Hercules CPF/East/West retained separately
# =====================================================================
def aggregate_pdp_stream(
    df,
    stream
):
    d = df.copy()
    if stream == "Oil":
        d["Output_Facility"] = (
            d["Facility"]
            .apply(oil_facility_map)
        )
        value_col = (
            "Net_Oil_BBL"
        )
    else:
        d["Output_Facility"] = (
            d["Facility"]
            .apply(gas_facility_map)
        )
        value_col = (
            "Net_Gas_MCF"
        )
    out = (
        d.groupby(
            [
                "Output_Facility",
                "Date"
            ],
            as_index=False
        )[value_col]
        .sum()
    )
    return out.rename(
        columns={
            "Output_Facility":
                "Facility"
        }
    )
pdp_high_oil = aggregate_pdp_stream(
    pdp_high_wells,
    "Oil"
)
pdp_high_gas = aggregate_pdp_stream(
    pdp_high_wells,
    "Gas"
)
pdp_low_oil = aggregate_pdp_stream(
    pdp_low_wells,
    "Oil"
)
pdp_low_gas = aggregate_pdp_stream(
    pdp_low_wells,
    "Gas"
)
# =====================================================================
# REMAP EXISTING CFD / UFD FACILITY AGGREGATES
#
# Existing notebook variables expected:
#
#   cfd_fac
#   ud_fac
#
# with:
#
#   Facility
#   Date
#   Net_Oil_BBL
#   Net_Gas_MCF
# =====================================================================
def remap_component(
    df,
    stream
):
    d = df.copy()
    d["Facility"] = (
        d["Facility"]
        .apply(rtp_normalize_facility)
    )
    if stream == "Oil":
        d["Facility"] = (
            d["Facility"]
            .apply(oil_facility_map)
        )
        value_col = (
            "Net_Oil_BBL"
        )
    else:
        d["Facility"] = (
            d["Facility"]
            .apply(gas_facility_map)
        )
        value_col = (
            "Net_Gas_MCF"
        )
    return (
        d.groupby(
            [
                "Facility",
                "Date"
            ],
            as_index=False
        )[value_col]
        .sum()
    )
cfd_oil_rtp = remap_component(
    cfd_fac,
    "Oil"
)
cfd_gas_rtp = remap_component(
    cfd_fac,
    "Gas"
)
ufd_oil_rtp = remap_component(
    ud_fac,
    "Oil"
)
ufd_gas_rtp = remap_component(
    ud_fac,
    "Gas"
)
# =====================================================================
# BUILD CASE 01 / 02 / 03
# =====================================================================
def build_rtp_case(
    pdp_df,
    stream,
    case_number
):
    if stream == "Oil":
        pdp_col = (
            "Net_Oil_BBL"
        )
        component_col = (
            "Net_Oil_BBL"
        )
        cfd_df = (
            cfd_oil_rtp
        )
        ufd_df = (
            ufd_oil_rtp
        )
    else:
        pdp_col = (
            "Net_Gas_MCF"
        )
        component_col = (
            "Net_Gas_MCF"
        )
        cfd_df = (
            cfd_gas_rtp
        )
        ufd_df = (
            ufd_gas_rtp
        )
    # -----------------------------------------------------------------
    # PDP
    # -----------------------------------------------------------------
    pdp_part = pdp_df[
        [
            "Facility",
            "Date",
            pdp_col
        ]
    ].rename(
        columns={
            pdp_col:
                "PDP_Volume"
        }
    )
    frames = [
        pdp_part
    ]
    # -----------------------------------------------------------------
    # CFD
    # -----------------------------------------------------------------
    if case_number >= 2:
        cfd_part = cfd_df[
            [
                "Facility",
                "Date",
                component_col
            ]
        ].rename(
            columns={
                component_col:
                    "CFD_Volume"
            }
        )
        frames.append(
            cfd_part
        )
    # -----------------------------------------------------------------
    # UFD
    # -----------------------------------------------------------------
    if case_number >= 3:
        ufd_part = ufd_df[
            [
                "Facility",
                "Date",
                component_col
            ]
        ].rename(
            columns={
                component_col:
                    "UFD_Volume"
            }
        )
        frames.append(
            ufd_part
        )
    # -----------------------------------------------------------------
    # FULL FACILITY / DATE KEY
    # -----------------------------------------------------------------
    keys = (
        pd.concat(
            [
                x[
                    [
                        "Facility",
                        "Date"
                    ]
                ]
                for x in frames
            ],
            ignore_index=True
        )
        .drop_duplicates()
    )
    out = keys.merge(
        pdp_part,
        on=[
            "Facility",
            "Date"
        ],
        how="left"
    )
    if case_number >= 2:
        out = out.merge(
            cfd_part,
            on=[
                "Facility",
                "Date"
            ],
            how="left"
        )
    else:
        out[
            "CFD_Volume"
        ] = 0.0
    if case_number >= 3:
        out = out.merge(
            ufd_part,
            on=[
                "Facility",
                "Date"
            ],
            how="left"
        )
    else:
        out[
            "UFD_Volume"
        ] = 0.0
    for col in [
        "PDP_Volume",
        "CFD_Volume",
        "UFD_Volume"
    ]:
        out[col] = (
            pd.to_numeric(
                out[col],
                errors="coerce"
            )
            .fillna(0.0)
        )
    out[
        "Total_Volume"
    ] = (
        out[
            "PDP_Volume"
        ]
        +
        out[
            "CFD_Volume"
        ]
        +
        out[
            "UFD_Volume"
        ]
    )
    out["Date"] = pd.to_datetime(
        out["Date"]
    )
    out[
        "Days_In_Month"
    ] = (
        out[
            "Date"
        ].dt.days_in_month
    )
    out[
        "Rate"
    ] = (
        out[
            "Total_Volume"
        ]
        /
        out[
            "Days_In_Month"
        ]
    )
    out = (
        out.sort_values(
            [
                "Facility",
                "Date"
            ]
        )
        .reset_index(
            drop=True
        )
    )
    out[
        "Cumulative"
    ] = (
        out.groupby(
            "Facility"
        )[
            "Total_Volume"
        ]
        .cumsum()
    )
    return out
# =====================================================================
# CREATE ALL HIGH / LOW CASES
# =====================================================================
cases = {}
for scenario, oil_pdp, gas_pdp in [
    (
        "HIGH_ON",
        pdp_high_oil,
        pdp_high_gas
    ),
    (
        "LOW_OFF",
        pdp_low_oil,
        pdp_low_gas
    )
]:
    for case_number in [
        1,
        2,
        3
    ]:
        cases[
            (
                scenario,
                case_number,
                "Oil"
            )
        ] = build_rtp_case(
            oil_pdp,
            "Oil",
            case_number
        )
        cases[
            (
                scenario,
                case_number,
                "Gas"
            )
        ] = build_rtp_case(
            gas_pdp,
            "Gas",
            case_number
        )
# =====================================================================
# OUTPUT NAMES
# =====================================================================
CASE_NAMES = {
    1:
        "PDP_Base",
    2:
        "PDP_Plus_Confirmed",
    3:
        "PDP_Plus_Confirmed_Plus_Uncertain"
}
# =====================================================================
# EXPORT ALL FACILITY-MONTH CSVs
# =====================================================================
def export_rtp_case(
    df,
    scenario,
    case_number,
    stream
):
    d = df.copy()
    if stream == "Oil":
        d = d.rename(
            columns={
                "PDP_Volume":
                    "PDP_Oil_BBL",
                "CFD_Volume":
                    "Confirmed_Oil_BBL",
                "UFD_Volume":
                    "Uncertain_Oil_BBL",
                "Total_Volume":
                    "Total_Oil_BBL",
                "Rate":
                    "Total_Oil_BPD",
                "Cumulative":
                    "Total_Oil_Cumulative_BBL"
            }
        )
    else:
        d = d.rename(
            columns={
                "PDP_Volume":
                    "PDP_Gas_MCF",
                "CFD_Volume":
                    "Confirmed_Gas_MCF",
                "UFD_Volume":
                    "Uncertain_Gas_MCF",
                "Total_Volume":
                    "Total_Gas_MCF",
                "Rate":
                    "Total_Gas_MCFD",
                "Cumulative":
                    "Total_Gas_Cumulative_MCF"
            }
        )
    filename = (
        f"RTP_{scenario}_"
        f"{case_number:02d}_"
        f"{CASE_NAMES[case_number]}_"
        f"{stream}.csv"
    )
    d.to_csv(
        filename,
        index=False
    )
    print(
        "Saved:",
        filename
    )
for scenario in [
    "HIGH_ON",
    "LOW_OFF"
]:
    for case_number in [
        1,
        2,
        3
    ]:
        for stream in [
            "Oil",
            "Gas"
        ]:
            export_rtp_case(
                cases[
                    (
                        scenario,
                        case_number,
                        stream
                    )
                ],
                scenario,
                case_number,
                stream
            )
# =====================================================================
# RECORDED JULY PDP ACTUAL RATE
#
# IMPORTANT:
#
# This intentionally uses ORIGINAL RECORDED PRODUCTION.
#
# It does NOT use the Problematic DSU replacement type curve.
#
# Therefore this metric answers:
#
# "What was actually recorded for PDPs on the latest production month?"
#
# rather than:
#
# "What does the HIGH modeled profile say?"
# =====================================================================
def build_recorded_actual_table(
    stream
):
    d = pdp_actual_rtp[
        pdp_actual_rtp[
            "Date"
        ]
        ==
        RECORDED_PRODUCTION_DATE
    ].copy()
    route = pdp_route_rtp[
        [
            "Facility",
            "Match_Name",
            "WI"
        ]
    ].copy()
    d = d.merge(
        route,
        on="Match_Name",
        how="inner"
    )
    if stream == "Oil":
        d[
            "Facility"
        ] = (
            d["Facility"]
            .apply(oil_facility_map)
        )
        d[
            "Net_Volume"
        ] = (
            d[
                "Oil_BBL"
            ]
            *
            d[
                "WI"
            ]
        )
    else:
        d[
            "Facility"
        ] = (
            d["Facility"]
            .apply(gas_facility_map)
        )
        d[
            "Net_Volume"
        ] = (
            d[
                "Gas_MCF"
            ]
            *
            d[
                "WI"
            ]
        )
    out = (
        d.groupby(
            "Facility",
            as_index=False
        )[
            "Net_Volume"
        ]
        .sum()
    )
    out[
        "Recorded_PDP_Rate_7_15_2026"
    ] = (
        out[
            "Net_Volume"
        ]
        /
        RECORDED_PRODUCTION_DATE.days_in_month
    )
    return out[
        [
            "Facility",
            "Recorded_PDP_Rate_7_15_2026"
        ]
    ]
recorded_oil = (
    build_recorded_actual_table(
        "Oil"
    )
)
recorded_oil[
    "Stream"
] = "Oil"
recorded_gas = (
    build_recorded_actual_table(
        "Gas"
    )
)
recorded_gas[
    "Stream"
] = "Gas"
recorded_all = pd.concat(
    [
        recorded_oil,
        recorded_gas
    ],
    ignore_index=True
)
# =====================================================================
# SUMMARY METRICS
# =====================================================================
def summarize_rtp_case(
    df,
    scenario,
    case_number,
    stream
):
    rows = []
    for facility, d in df.groupby(
        "Facility"
    ):
        d = (
            d.sort_values(
                "Date"
            )
            .copy()
        )
        # -------------------------------------------------------------
        # FORECAST-DEPENDENT CURRENT RATE
        # -------------------------------------------------------------
        current = d[
            d["Date"]
            ==
            FORECAST_CURRENT_DATE
        ]
        if len(current) > 0:
            current_rate = float(
                current.iloc[0][
                    "Rate"
                ]
            )
        else:
            current_rate = np.nan
        # -------------------------------------------------------------
        # RATE ON 2/1/2034
        # -------------------------------------------------------------
        target = d[
            d["Date"]
            ==
            TARGET_2034_DATE
        ]
        if len(target) > 0:
            rate_2034 = float(
                target.iloc[0][
                    "Rate"
                ]
            )
        else:
            rate_2034 = np.nan
        # -------------------------------------------------------------
        # PEAK ON / AFTER 2/1/2034
        # -------------------------------------------------------------
        post = d[
            d["Date"]
            >=
            TARGET_2034_DATE
        ].copy()
        if len(post) > 0:
            peak_index = (
                post[
                    "Rate"
                ].idxmax()
            )
            peak_rate = float(
                post.loc[
                    peak_index,
                    "Rate"
                ]
            )
            peak_date = (
                post.loc[
                    peak_index,
                    "Date"
                ]
            )
        else:
            peak_rate = np.nan
            peak_date = pd.NaT
        # -------------------------------------------------------------
        # ENDING CUMULATIVE
        # -------------------------------------------------------------
        if len(d) > 0:
            end_date = (
                d.iloc[-1][
                    "Date"
                ]
            )
            end_cumulative = float(
                d.iloc[-1][
                    "Cumulative"
                ]
            )
        else:
            end_date = pd.NaT
            end_cumulative = np.nan
        rows.append(
            {
                "Facility":
                    facility,
                "Stream":
                    stream,
                "Case_Number":
                    case_number,
                "Case":
                    CASE_NAMES[
                        case_number
                    ],
                "RTP_Scenario":
                    (
                        "HIGH / ON"
                        if scenario
                        ==
                        "HIGH_ON"
                        else
                        "LOW / OFF"
                    ),
                "Forecast_Dependent_Current_Date":
                    FORECAST_CURRENT_DATE,
                "Forecast_Dependent_Current_Rate":
                    current_rate,
                "Rate_2_1_2034":
                    rate_2034,
                "Post_2_1_2034_Peak_Rate":
                    peak_rate,
                "Post_2_1_2034_Peak_Date":
                    peak_date,
                "End_Date":
                    end_date,
                "End_Cumulative":
                    end_cumulative
            }
        )
    return pd.DataFrame(
        rows
    )
summary_parts = []
for scenario in [
    "HIGH_ON",
    "LOW_OFF"
]:
    for case_number in [
        1,
        2,
        3
    ]:
        for stream in [
            "Oil",
            "Gas"
        ]:
            summary_parts.append(
                summarize_rtp_case(
                    cases[
                        (
                            scenario,
                            case_number,
                            stream
                        )
                    ],
                    scenario,
                    case_number,
                    stream
                )
            )
summary_long = pd.concat(
    summary_parts,
    ignore_index=True
)
# =====================================================================
# ADD RECORDED JULY ACTUAL REFERENCE
# =====================================================================
summary_long = summary_long.merge(
    recorded_all,
    on=[
        "Facility",
        "Stream"
    ],
    how="left"
)
summary_long[
    "Recorded_PDP_Production_Date"
] = RECORDED_PRODUCTION_DATE
# =====================================================================
# HIGH / LOW SIDE-BY-SIDE
# =====================================================================
key_cols = [
    "Facility",
    "Stream",
    "Case_Number",
    "Case"
]
high = summary_long[
    summary_long[
        "RTP_Scenario"
    ]
    ==
    "HIGH / ON"
].copy()
low = summary_long[
    summary_long[
        "RTP_Scenario"
    ]
    ==
    "LOW / OFF"
].copy()
high = high[
    key_cols
    +
    [
        "Forecast_Dependent_Current_Date",
        "Recorded_PDP_Production_Date",
        "Recorded_PDP_Rate_7_15_2026",
        "Forecast_Dependent_Current_Rate",
        "Rate_2_1_2034",
        "Post_2_1_2034_Peak_Rate",
        "Post_2_1_2034_Peak_Date",
        "End_Date",
        "End_Cumulative"
    ]
]
high = high.rename(
    columns={
        "Forecast_Dependent_Current_Rate":
            "HIGH_ON_Current_Rate",
        "Rate_2_1_2034":
            "HIGH_ON_Rate_2_1_2034",
        "Post_2_1_2034_Peak_Rate":
            "HIGH_ON_Post2034_Peak_Rate",
        "Post_2_1_2034_Peak_Date":
            "HIGH_ON_Post2034_Peak_Date",
        "End_Date":
            "HIGH_ON_End_Date",
        "End_Cumulative":
            "HIGH_ON_End_Cumulative"
    }
)
low = low[
    key_cols
    +
    [
        "Forecast_Dependent_Current_Rate",
        "Rate_2_1_2034",
        "Post_2_1_2034_Peak_Rate",
        "Post_2_1_2034_Peak_Date",
        "End_Date",
        "End_Cumulative"
    ]
]
low = low.rename(
    columns={
        "Forecast_Dependent_Current_Rate":
            "LOW_OFF_Current_Rate",
        "Rate_2_1_2034":
            "LOW_OFF_Rate_2_1_2034",
        "Post_2_1_2034_Peak_Rate":
            "LOW_OFF_Post2034_Peak_Rate",
        "Post_2_1_2034_Peak_Date":
            "LOW_OFF_Post2034_Peak_Date",
        "End_Date":
            "LOW_OFF_End_Date",
        "End_Cumulative":
            "LOW_OFF_End_Cumulative"
    }
)
summary = high.merge(
    low,
    on=key_cols,
    how="outer"
)
# =====================================================================
# HIGH MINUS LOW DELTAS
#
# Positive = HIGH / ON adds rate or cumulative.
# =====================================================================
summary[
    "Delta_Current_Rate_HIGH_minus_LOW"
] = (
    summary[
        "HIGH_ON_Current_Rate"
    ]
    -
    summary[
        "LOW_OFF_Current_Rate"
    ]
)
summary[
    "Delta_Rate_2_1_2034_HIGH_minus_LOW"
] = (
    summary[
        "HIGH_ON_Rate_2_1_2034"
    ]
    -
    summary[
        "LOW_OFF_Rate_2_1_2034"
    ]
)
summary[
    "Delta_Post2034_Peak_Rate_HIGH_minus_LOW"
] = (
    summary[
        "HIGH_ON_Post2034_Peak_Rate"
    ]
    -
    summary[
        "LOW_OFF_Post2034_Peak_Rate"
    ]
)
summary[
    "Delta_End_Cumulative_HIGH_minus_LOW"
] = (
    summary[
        "HIGH_ON_End_Cumulative"
    ]
    -
    summary[
        "LOW_OFF_End_Cumulative"
    ]
)
# =====================================================================
# CASE 03 RANKINGS
#
# Oil and Gas independently.
# Rank 1 = highest.
# =====================================================================
aggressive = summary[
    summary[
        "Case_Number"
    ]
    ==
    3
].copy()
for scenario_prefix in [
    "HIGH_ON",
    "LOW_OFF"
]:
    aggressive[
        f"{scenario_prefix}_Current_Rank"
    ] = (
        aggressive.groupby(
            "Stream"
        )[
            f"{scenario_prefix}_Current_Rate"
        ]
        .rank(
            method="min",
            ascending=False
        )
        .astype(
            "Int64"
        )
    )
    aggressive[
        f"{scenario_prefix}_2_1_2034_Rank"
    ] = (
        aggressive.groupby(
            "Stream"
        )[
            f"{scenario_prefix}_Rate_2_1_2034"
        ]
        .rank(
            method="min",
            ascending=False
        )
        .astype(
            "Int64"
        )
    )
    aggressive[
        f"{scenario_prefix}_Post2034_Peak_Rank"
    ] = (
        aggressive.groupby(
            "Stream"
        )[
            f"{scenario_prefix}_Post2034_Peak_Rate"
        ]
        .rank(
            method="min",
            ascending=False
        )
        .astype(
            "Int64"
        )
    )
rank_cols = [
    "Facility",
    "Stream",
    "HIGH_ON_Current_Rank",
    "LOW_OFF_Current_Rank",
    "HIGH_ON_2_1_2034_Rank",
    "LOW_OFF_2_1_2034_Rank",
    "HIGH_ON_Post2034_Peak_Rank",
    "LOW_OFF_Post2034_Peak_Rank"
]
summary = summary.merge(
    aggressive[
        rank_cols
    ],
    on=[
        "Facility",
        "Stream"
    ],
    how="left"
)
# =====================================================================
# UNITS
# =====================================================================
summary[
    "Rate_Unit"
] = np.where(
    summary[
        "Stream"
    ]
    ==
    "Oil",
    "BPD",
    "MCF/D"
)
summary[
    "Cumulative_Unit"
] = np.where(
    summary[
        "Stream"
    ]
    ==
    "Oil",
    "BBL",
    "MCF"
)
# =====================================================================
# FINAL COLUMN ORDER
# =====================================================================
summary = summary[
    [
        "Facility",
        "Stream",
        "Rate_Unit",
        "Cumulative_Unit",
        "Case_Number",
        "Case",
        # -------------------------------------------------------------
        # RECORDED ACTUAL REFERENCE
        # -------------------------------------------------------------
        "Recorded_PDP_Production_Date",
        "Recorded_PDP_Rate_7_15_2026",
        # -------------------------------------------------------------
        # FORECAST-DEPENDENT CURRENT
        # -------------------------------------------------------------
        "Forecast_Dependent_Current_Date",
        "HIGH_ON_Current_Rate",
        "LOW_OFF_Current_Rate",
        "Delta_Current_Rate_HIGH_minus_LOW",
        "HIGH_ON_Current_Rank",
        "LOW_OFF_Current_Rank",
        # -------------------------------------------------------------
        # 2/1/2034
        # -------------------------------------------------------------
        "HIGH_ON_Rate_2_1_2034",
        "LOW_OFF_Rate_2_1_2034",
        "Delta_Rate_2_1_2034_HIGH_minus_LOW",
        "HIGH_ON_2_1_2034_Rank",
        "LOW_OFF_2_1_2034_Rank",
        # -------------------------------------------------------------
        # POST-2034 PEAK
        # -------------------------------------------------------------
        "HIGH_ON_Post2034_Peak_Rate",
        "HIGH_ON_Post2034_Peak_Date",
        "LOW_OFF_Post2034_Peak_Rate",
        "LOW_OFF_Post2034_Peak_Date",
        "Delta_Post2034_Peak_Rate_HIGH_minus_LOW",
        "HIGH_ON_Post2034_Peak_Rank",
        "LOW_OFF_Post2034_Peak_Rank",
        # -------------------------------------------------------------
        # CUMULATIVE
        # -------------------------------------------------------------
        "HIGH_ON_End_Date",
        "HIGH_ON_End_Cumulative",
        "LOW_OFF_End_Date",
        "LOW_OFF_End_Cumulative",
        "Delta_End_Cumulative_HIGH_minus_LOW"
    ]
]
summary = (
    summary.sort_values(
        [
            "Stream",
            "Facility",
            "Case_Number"
        ]
    )
    .reset_index(
        drop=True
    )
)
# =====================================================================
# SAVE MASTER SUMMARY
# =====================================================================
summary.to_csv(
    SUMMARY_FILE,
    index=False
)
# =====================================================================
# FINISH
# =====================================================================
print()
print(
    "============================================================"
)
print(
    "PDP RETURN-TO-PRODUCTION (RTP) SENSITIVITY COMPLETE"
)
print(
    "============================================================"
)
print()
print(
    "HIGH / ON:"
)
print(
    "  Normal PDPs = actual + massive PDP forecast"
)
print(
    "  Sensitivity Wells = actual + massive PDP forecast"
)
print(
    "  Problematic DSUs = TYPE CURVE ONLY"
)
print(
    "                     Original production completely removed"
)
print(
    "                     Original PDP forecast completely removed"
)
print()
print(
    "LOW / OFF:"
)
print(
    "  Normal PDPs = actual + massive PDP forecast"
)
print(
    "  Sensitivity Wells = actual then ZERO"
)
print(
    "  Problematic DSUs = actual + massive PDP forecast"
)
print()
print(
    "Problematic DSU HIGH-case TC alignment:"
)
print(
    "  Forecast Month 1 = original first production month"
)
print(
    "  (first PDP forecast month only if no actual production exists)"
)
print()
print(
    "Forecast-dependent current date:",
    FORECAST_CURRENT_DATE.strftime(
        "%m/%d/%Y"
    )
)
print(
    "Recorded PDP actual reference date:",
    RECORDED_PRODUCTION_DATE.strftime(
        "%m/%d/%Y"
    )
)
print()
print(
    "IMPORTANT:"
)
print(
    "  Recorded_PDP_Rate_7_15_2026 is always the REAL recorded"
)
print(
    "  production value, even though Problematic DSU production"
)
print(
    "  is replaced by the TC in the HIGH modeled case."
)
print()
print(
    "Hercules Oil:"
)
print(
    "  CPF + East Satellite + West Satellite combined as HERCULES"
)
print()
print(
    "Hercules Gas:"
)
print(
    "  CPF / East Satellite / West Satellite remain separate"
)
print()
print(
    "Delta convention = HIGH / ON - LOW / OFF"
)
print()
print(
    "Saved:",
    SUMMARY_FILE
)
print()
display(
    summary
)


In [ ]:
# ============================================================
# 8/15/2026 TOTAL CURRENT PRODUCTION BY FACILITY
#
# Source:
#   One combined current-production CSV containing all wells
#
# File layout:
#   Col A = Well Name
#   Col H = Date
#   Col I = Oil (BBL/M)
#   Col J = Gas (MCF/M)
#
# Routing/WI:
#   Uses already-established:
#       pdp_route
#       CFD routing dataframe
#       UFD routing dataframe
#
# Output:
#   08_15_2026_Total_Production_By_Facility.csv
#
# Hercules:
#   OIL -> CPF + EAST + WEST = HERCULES
#   GAS -> CPF / EAST / WEST remain separate
# ============================================================
import pandas as pd
import numpy as np
import re
# ------------------------------------------------------------
# CHANGE THIS TO YOUR UPLOADED CSV NAME
# ------------------------------------------------------------
ALL_CURRENT_PRODUCTION_FILE = "All production current.csv"
TARGET_DATE = pd.Timestamp("2026-08-15")
OUTPUT_FILE = "08_15_2026_Total_Production_By_Facility.csv"
# ============================================================
# NORMALIZATION
# ============================================================
def prod_norm_name(v):
    if pd.isna(v):
        return ""
    return re.sub(
        r"\s+",
        " ",
        str(v).strip()
    ).upper()
def prod_norm_facility(v):
    if pd.isna(v):
        return ""
    return re.sub(
        r"\s+",
        " ",
        str(v).strip()
    ).upper()
HERCULES_NAMES = {
    "HERCULES CPF",
    "HERCULES EAST SATELLITE",
    "HERCULES WEST SATELLITE"
}
def oil_facility_map(v):
    f = prod_norm_facility(v)
    if f in HERCULES_NAMES:
        return "HERCULES"
    return f
def gas_facility_map(v):
    return prod_norm_facility(v)
# ============================================================
# HELPER: STANDARDIZE WI
# ============================================================
def clean_wi(series):
    x = pd.to_numeric(
        series,
        errors="coerce"
    )
    # If WI appears to be stored as percent, convert to decimal
    if x.dropna().shape[0] > 0:
        if x.dropna().max() > 1:
            x = x / 100.0
    return x
# ============================================================
# BUILD ONE MASTER ROUTING TABLE
#
# PDP routing is known as pdp_route.
#
# This section looks for the existing CFD/UFD routing
# dataframes by common names so you don't have to reread files.
# ============================================================
# ------------------------------------------------------------
# PDP
# ------------------------------------------------------------
pdp_r = pdp_route.copy()
if "Match_Name" not in pdp_r.columns:
    # Find likely well-name column
    for c in [
        "Well_Name",
        "Well Name",
        "Well"
    ]:
        if c in pdp_r.columns:
            pdp_r["Match_Name"] = (
                pdp_r[c]
                .apply(prod_norm_name)
            )
            break
else:
    pdp_r["Match_Name"] = (
        pdp_r["Match_Name"]
        .apply(prod_norm_name)
    )
pdp_r["Facility"] = (
    pdp_r["Facility"]
    .apply(prod_norm_facility)
)
pdp_r["WI"] = clean_wi(
    pdp_r["WI"]
)
pdp_r = pdp_r[
    [
        "Match_Name",
        "Facility",
        "WI"
    ]
].copy()
pdp_r["Category"] = "PDP"
# ============================================================
# FIND CFD ROUTING DATAFRAME
# ============================================================
cfd_route_existing = None
for name in [
    "cfd_route",
    "cfd_routing",
    "CFD_route",
    "CFD_ROUTING"
]:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame):
            cfd_route_existing = obj.copy()
            break
if cfd_route_existing is None:
    raise ValueError(
        "Could not find the existing CFD routing dataframe. "
        "Expected something like cfd_route."
    )
cfd_r = cfd_route_existing.copy()
# Find CFD well-name column
cfd_well_col = None
for c in [
    "Match_Name",
    "Well_Name",
    "Well Name",
    "Well"
]:
    if c in cfd_r.columns:
        cfd_well_col = c
        break
if cfd_well_col is None:
    raise ValueError(
        "Could not identify CFD well-name column."
    )
cfd_r["Match_Name"] = (
    cfd_r[cfd_well_col]
    .apply(prod_norm_name)
)
cfd_r["Facility"] = (
    cfd_r["Facility"]
    .apply(prod_norm_facility)
)
# Find CFD WI column
cfd_wi_col = None
for c in [
    "WI",
    "WI %",
    "WI%",
    "Working Interest"
]:
    if c in cfd_r.columns:
        cfd_wi_col = c
        break
if cfd_wi_col is None:
    raise ValueError(
        "Could not identify CFD WI column."
    )
cfd_r["WI"] = clean_wi(
    cfd_r[cfd_wi_col]
)
cfd_r = cfd_r[
    [
        "Match_Name",
        "Facility",
        "WI"
    ]
].copy()
cfd_r["Category"] = "CFD"
# ============================================================
# FIND UFD ROUTING DATAFRAME
# ============================================================
ufd_route_existing = None
for name in [
    "ufd_route",
    "ud_route",
    "ufd_routing",
    "UFD_route",
    "UFD_ROUTING"
]:
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame):
            ufd_route_existing = obj.copy()
            break
if ufd_route_existing is None:
    raise ValueError(
        "Could not find the existing UFD routing dataframe. "
        "Expected something like ufd_route or ud_route."
    )
ufd_r = ufd_route_existing.copy()
# Find UFD well-name column
ufd_well_col = None
for c in [
    "Match_Name",
    "Well_Name",
    "Well Name",
    "Well"
]:
    if c in ufd_r.columns:
        ufd_well_col = c
        break
if ufd_well_col is None:
    raise ValueError(
        "Could not identify UFD well-name column."
    )
ufd_r["Match_Name"] = (
    ufd_r[ufd_well_col]
    .apply(prod_norm_name)
)
ufd_r["Facility"] = (
    ufd_r["Facility"]
    .apply(prod_norm_facility)
)
# Find UFD WI column
ufd_wi_col = None
for c in [
    "WI",
    "WI %",
    "WI%",
    "Working Interest"
]:
    if c in ufd_r.columns:
        ufd_wi_col = c
        break
if ufd_wi_col is None:
    raise ValueError(
        "Could not identify UFD WI column."
    )
ufd_r["WI"] = clean_wi(
    ufd_r[ufd_wi_col]
)
ufd_r = ufd_r[
    [
        "Match_Name",
        "Facility",
        "WI"
    ]
].copy()
ufd_r["Category"] = "UFD"
# ============================================================
# COMBINE ALL ROUTINGS
# ============================================================
routing_all = pd.concat(
    [
        pdp_r,
        cfd_r,
        ufd_r
    ],
    ignore_index=True
)
# Drop exact duplicate routing rows
routing_all = routing_all.drop_duplicates(
    subset=[
        "Match_Name",
        "Facility",
        "WI",
        "Category"
    ]
)
# ============================================================
# SAFETY CHECK:
# A well should not route to multiple categories/facilities.
# ============================================================
route_check = (
    routing_all
    .groupby("Match_Name")
    .agg(
        Facility_Count=("Facility", "nunique"),
        Category_Count=("Category", "nunique")
    )
)
bad_routes = route_check[
    (route_check["Facility_Count"] > 1)
    |
    (route_check["Category_Count"] > 1)
]
if len(bad_routes) > 0:
    print(
        "WARNING:",
        len(bad_routes),
        "well names appear in more than one routing/category."
    )
    display(
        bad_routes.head(20)
    )
# ============================================================
# READ 8/15/2026 PRODUCTION FILE
# ============================================================
prod_parts = []
for chunk in pd.read_csv(
    ALL_CURRENT_PRODUCTION_FILE,
    usecols=[0, 7, 8, 9],
    chunksize=200000
):
    chunk.columns = [
        "Well_Name",
        "Date",
        "Oil_BBL",
        "Gas_MCF"
    ]
    chunk["Match_Name"] = (
        chunk["Well_Name"]
        .apply(prod_norm_name)
    )
    chunk["Date"] = pd.to_datetime(
        chunk["Date"],
        errors="coerce"
    )
    # EXACT 8/15/2026
    chunk = chunk[
        chunk["Date"] == TARGET_DATE
    ].copy()
    if len(chunk) == 0:
        continue
    chunk["Oil_BBL"] = pd.to_numeric(
        chunk["Oil_BBL"],
        errors="coerce"
    ).fillna(0.0)
    chunk["Gas_MCF"] = pd.to_numeric(
        chunk["Gas_MCF"],
        errors="coerce"
    ).fillna(0.0)
    prod_parts.append(
        chunk[
            [
                "Match_Name",
                "Oil_BBL",
                "Gas_MCF"
            ]
        ]
    )
if not prod_parts:
    raise ValueError(
        "No 8/15/2026 rows were found in "
        f"{ALL_CURRENT_PRODUCTION_FILE}"
    )
prod_815 = pd.concat(
    prod_parts,
    ignore_index=True
)
# If duplicate rows exist for same well/date,
# sum them before routing.
prod_815 = (
    prod_815
    .groupby(
        "Match_Name",
        as_index=False
    )[
        [
            "Oil_BBL",
            "Gas_MCF"
        ]
    ]
    .sum()
)
# ============================================================
# ROUTE + APPLY WI
# ============================================================
prod_net = prod_815.merge(
    routing_all,
    on="Match_Name",
    how="inner"
)
prod_net["Net_Oil_BBL"] = (
    prod_net["Oil_BBL"]
    *
    prod_net["WI"]
)
prod_net["Net_Gas_MCF"] = (
    prod_net["Gas_MCF"]
    *
    prod_net["WI"]
)
# ============================================================
# OIL AGGREGATION
#
# Hercules combined
# ============================================================
oil = prod_net[
    [
        "Facility",
        "Net_Oil_BBL"
    ]
].copy()
oil["Facility"] = (
    oil["Facility"]
    .apply(oil_facility_map)
)
oil = (
    oil
    .groupby(
        "Facility",
        as_index=False
    )["Net_Oil_BBL"]
    .sum()
    .rename(
        columns={
            "Net_Oil_BBL":
                "Net_Oil_BBL_Month"
        }
    )
)
# August has 31 days
oil["Net_Oil_BPD"] = (
    oil["Net_Oil_BBL_Month"]
    / 31.0
)
# ============================================================
# GAS AGGREGATION
#
# Hercules stays split
# ============================================================
gas = prod_net[
    [
        "Facility",
        "Net_Gas_MCF"
    ]
].copy()
gas["Facility"] = (
    gas["Facility"]
    .apply(gas_facility_map)
)
gas = (
    gas
    .groupby(
        "Facility",
        as_index=False
    )["Net_Gas_MCF"]
    .sum()
    .rename(
        columns={
            "Net_Gas_MCF":
                "Net_Gas_MCF_Month"
        }
    )
)
gas["Net_Gas_MCFD"] = (
    gas["Net_Gas_MCF_Month"]
    / 31.0
)
# ============================================================
# OUTPUT
#
# Because Hercules oil is combined but gas is split:
#
#   HERCULES row contains combined Hercules oil
#
#   HERCULES CPF
#   HERCULES EAST SATELLITE
#   HERCULES WEST SATELLITE
#
# contain their respective gas volumes.
# ============================================================
facility_total = pd.merge(
    oil,
    gas,
    on="Facility",
    how="outer"
)
for c in [
    "Net_Oil_BBL_Month",
    "Net_Oil_BPD",
    "Net_Gas_MCF_Month",
    "Net_Gas_MCFD"
]:
    facility_total[c] = (
        pd.to_numeric(
            facility_total[c],
            errors="coerce"
        )
        .fillna(0.0)
    )
# ============================================================
# RANKS
# ============================================================
facility_total["Oil_Rank"] = (
    facility_total["Net_Oil_BPD"]
    .rank(
        method="min",
        ascending=False
    )
    .astype("Int64")
)
facility_total["Gas_Rank"] = (
    facility_total["Net_Gas_MCFD"]
    .rank(
        method="min",
        ascending=False
    )
    .astype("Int64")
)
# ============================================================
# SORT
# ============================================================
facility_total = (
    facility_total
    .sort_values(
        [
            "Oil_Rank",
            "Gas_Rank"
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)
# ============================================================
# SAVE + DISPLAY
# ============================================================
facility_total.to_csv(
    OUTPUT_FILE,
    index=False
)
print("=" * 70)
print("8/15/2026 NET PRODUCTION BY FACILITY")
print("=" * 70)
print(
    "Source:",
    ALL_CURRENT_PRODUCTION_FILE
)
print(
    "PDP + CFD + UFD wells routed using existing routing tables."
)
print(
    "WI applied before facility aggregation."
)
print(
    "Oil: Hercules CPF + East + West combined as HERCULES."
)
print(
    "Gas: Hercules CPF + East + West remain separate."
)
print(
    "\nProduction wells in 8/15/2026 source file:",
    prod_815["Match_Name"].nunique()
)
print(
    "Production wells successfully matched to routing:",
    prod_net["Match_Name"].nunique()
)
print(
    "\nSaved:",
    OUTPUT_FILE
)
display(facility_total)


In [ ]:
# ============================================================
# 8/15/2026 ZERO / MISSING PRODUCTION DIAGNOSTIC
#
# CATEGORY A = ZERO
#   Well has an 8/15/2026 production record,
#   but BOTH Oil = 0 and Gas = 0.
#
# CATEGORY B = MISSING
#   Well exists in routing but has NO 8/15/2026
#   production record in All production current.csv.
#
# Uses:
#   routing_all  -> PDP + CFD + UFD routing population
#   prod_815     -> 8/15/2026 production records
#
# Outputs:
#   08_15_2026_Zero_or_Missing_Production_Wells.csv
# ============================================================
import pandas as pd
OUTPUT_FILE = (
    "08_15_2026_Zero_or_Missing_Production_Wells.csv"
)
# ============================================================
# 1. CREATE ONE ROW PER ROUTED WELL
# ============================================================
routes_check = routing_all.copy()
# If a well appears in more than one category,
# retain all category labels in the diagnostic.
route_summary = (
    routes_check
    .groupby(
        "Match_Name",
        as_index=False
    )
    .agg(
        Facility=(
            "Facility",
            lambda x: " | ".join(
                sorted(set(x.astype(str)))
            )
        ),
        Routing_Category=(
            "Category",
            lambda x: " | ".join(
                sorted(set(x.astype(str)))
            )
        ),
        WI=(
            "WI",
            "first"
        )
    )
)
# ============================================================
# 2. ENSURE ONE 8/15/2026 ROW PER PRODUCTION WELL
# ============================================================
prod_check = (
    prod_815
    .groupby(
        "Match_Name",
        as_index=False
    )[
        [
            "Oil_BBL",
            "Gas_MCF"
        ]
    ]
    .sum()
)
# Flag that an actual 8/15 record exists
prod_check["Has_8_15_Record"] = True
# ============================================================
# 3. MERGE FULL ROUTING POPULATION TO AUGUST PRODUCTION
# ============================================================
check = route_summary.merge(
    prod_check,
    on="Match_Name",
    how="left"
)
check["Has_8_15_Record"] = (
    check["Has_8_15_Record"]
    .fillna(False)
)
# ============================================================
# 4. IDENTIFY ISSUE TYPE
# ============================================================
# ----------------------------
# CATEGORY B = MISSING
# ----------------------------
missing_mask = (
    check["Has_8_15_Record"] == False
)
# ----------------------------
# CATEGORY A = ZERO
#
# Requires a record to exist,
# AND both oil and gas = zero.
# ----------------------------
zero_mask = (
    (check["Has_8_15_Record"] == True)
    &
    (check["Oil_BBL"].fillna(0) == 0)
    &
    (check["Gas_MCF"].fillna(0) == 0)
)
check["Issue_Category"] = ""
check.loc[
    zero_mask,
    "Issue_Category"
] = "A - ZERO"
check.loc[
    missing_mask,
    "Issue_Category"
] = "B - MISSING"
# ============================================================
# 5. KEEP ONLY PROBLEM WELLS
# ============================================================
issues = check[
    check["Issue_Category"] != ""
].copy()
# For missing rows, leave production blank rather than
# replacing with zero. This preserves the distinction
# between ZERO and MISSING.
issues.loc[
    issues["Issue_Category"] == "B - MISSING",
    ["Oil_BBL", "Gas_MCF"]
] = pd.NA
# ============================================================
# 6. FRIENDLY WELL NAME
# ============================================================
issues = issues.rename(
    columns={
        "Match_Name": "Well_Name",
        "Oil_BBL": "Aug_15_Oil_BBL",
        "Gas_MCF": "Aug_15_Gas_MCF"
    }
)
# ============================================================
# 7. COLUMN ORDER
# ============================================================
issues = issues[
    [
        "Issue_Category",
        "Well_Name",
        "Routing_Category",
        "Facility",
        "WI",
        "Aug_15_Oil_BBL",
        "Aug_15_Gas_MCF"
    ]
]
issues = (
    issues
    .sort_values(
        [
            "Issue_Category",
            "Routing_Category",
            "Facility",
            "Well_Name"
        ]
    )
    .reset_index(drop=True)
)
# ============================================================
# 8. SAVE
# ============================================================
issues.to_csv(
    OUTPUT_FILE,
    index=False
)
# ============================================================
# 9. SEPARATE A AND B LISTS
# ============================================================
zero_wells = issues[
    issues["Issue_Category"] == "A - ZERO"
].copy()
missing_wells = issues[
    issues["Issue_Category"] == "B - MISSING"
].copy()
# ============================================================
# 10. SUMMARY
# ============================================================
print("=" * 70)
print("8/15/2026 ZERO / MISSING PRODUCTION CHECK")
print("=" * 70)
print(
    "\nTotal routed unique wells:",
    route_summary["Match_Name"].nunique()
)
print(
    "Wells with an 8/15/2026 production record:",
    check["Has_8_15_Record"].sum()
)
print(
    "\nA - ZERO wells:",
    len(zero_wells)
)
print(
    "B - MISSING wells:",
    len(missing_wells)
)
print(
    "Total wells with an issue:",
    len(issues)
)
print(
    "\nSaved:",
    OUTPUT_FILE
)
# ============================================================
# 11. DISPLAY FULL ISSUE TABLE
# ============================================================
display(issues)
# Optional separate displays
print("\nA - ZERO PRODUCTION:")
display(zero_wells)
print("\nB - MISSING PRODUCTION:")
display(missing_wells)


In [ ]:
# ============================================================
# 4. IDENTIFY ISSUE TYPE
# ============================================================
# August has 31 days
check["Oil_BPD"] = (
    check["Oil_BBL"] / 31.0
)
check["Gas_MCFD"] = (
    check["Gas_MCF"] / 31.0
)
# ----------------------------
# CATEGORY B = MISSING
# ----------------------------
missing_mask = (
    check["Has_8_15_Record"] == False
)
# ----------------------------
# CATEGORY A = ZERO
#
# Record exists AND both oil and gas are zero
# ----------------------------
zero_mask = (
    (check["Has_8_15_Record"] == True)
    &
    (check["Oil_BBL"].fillna(0) == 0)
    &
    (check["Gas_MCF"].fillna(0) == 0)
)
# ----------------------------
# CATEGORY C = LOW OIL / NO GAS
#
# OR condition:
#   Oil < 10 BPD
#   OR
#   Gas = 0 / not reporting
#
# Only applies if not already A or B
# ----------------------------
low_oil_or_no_gas_mask = (
    (check["Has_8_15_Record"] == True)
    &
    (
        (check["Oil_BPD"].fillna(0) < 10)
        |
        (check["Gas_MCF"].fillna(0) <= 0)
    )
    &
    (~zero_mask)
)
check["Issue_Category"] = ""
check.loc[
    zero_mask,
    "Issue_Category"
] = "A - ZERO"
check.loc[
    missing_mask,
    "Issue_Category"
] = "B - MISSING"
check.loc[
    low_oil_or_no_gas_mask,
    "Issue_Category"
] = "C - LOW OIL / NO GAS"



In [ ]:
# ============================================================
# 8/15/2026 PRODUCTION ISSUE DIAGNOSTIC
#
# A - ZERO
#     Has 8/15/2026 record
#     AND Oil = 0 AND Gas = 0
#
# B - MISSING
#     Routed well has NO 8/15/2026 production record
#
# C - LOW OIL / NO GAS
#     Has 8/15/2026 record
#     AND:
#         Oil < 10 BPD
#         OR
#         Gas = 0 / not reporting
#
# True zero wells are classified as A, not C.
#
# Uses objects already created in prior cell:
#     routing_all
#     prod_815
#
# Output:
#     08_15_2026_Production_Issues.csv
# ============================================================
import pandas as pd
OUTPUT_FILE = "08_15_2026_Production_Issues.csv"
DAYS_IN_AUGUST = 31
# ============================================================
# 1. CREATE ONE ROW PER ROUTED WELL
# ============================================================
routes_check = routing_all.copy()
# Preserve multiple category/facility labels if a duplicate
# routing exists (for example, the Teton duplicate).
route_summary = (
    routes_check
    .groupby(
        "Match_Name",
        as_index=False
    )
    .agg(
        Facility=(
            "Facility",
            lambda x: " | ".join(
                sorted(set(x.dropna().astype(str)))
            )
        ),
        Routing_Category=(
            "Category",
            lambda x: " | ".join(
                sorted(set(x.dropna().astype(str)))
            )
        ),
        WI=(
            "WI",
            "first"
        )
    )
)
# ============================================================
# 2. CREATE ONE 8/15/2026 PRODUCTION ROW PER WELL
# ============================================================
prod_check = (
    prod_815
    .groupby(
        "Match_Name",
        as_index=False
    )[
        [
            "Oil_BBL",
            "Gas_MCF"
        ]
    ]
    .sum()
)
# Explicit flag tells us whether a production record existed.
# This is important so MISSING is not confused with ZERO.
prod_check["Has_8_15_Record"] = True
# ============================================================
# 3. MERGE ROUTED WELL POPULATION TO PRODUCTION
# ============================================================
check = route_summary.merge(
    prod_check,
    on="Match_Name",
    how="left"
)
check["Has_8_15_Record"] = (
    check["Has_8_15_Record"]
    .fillna(False)
    .astype(bool)
)
# ============================================================
# 4. CALCULATE DAILY RATES
# ============================================================
check["Oil_BPD"] = (
    check["Oil_BBL"]
    / DAYS_IN_AUGUST
)
check["Gas_MCFD"] = (
    check["Gas_MCF"]
    / DAYS_IN_AUGUST
)
# ============================================================
# 5. DEFINE ISSUE CATEGORIES
# ============================================================
# ------------------------------------------------------------
# B - MISSING
#
# Well exists in routing but there is no 8/15/2026
# production record in the source file.
# ------------------------------------------------------------
missing_mask = (
    check["Has_8_15_Record"] == False
)
# ------------------------------------------------------------
# A - ZERO
#
# Production record exists but BOTH oil and gas are zero.
# ------------------------------------------------------------
zero_mask = (
    (check["Has_8_15_Record"] == True)
    &
    (check["Oil_BBL"].fillna(0) == 0)
    &
    (check["Gas_MCF"].fillna(0) == 0)
)
# ------------------------------------------------------------
# C - LOW OIL / NO GAS
#
# OR LOGIC:
#
#     Oil < 10 BPD
#              OR
#     Gas <= 0 MCF
#
# Excludes true zero wells because those belong in A.
# ------------------------------------------------------------
low_oil_or_no_gas_mask = (
    (check["Has_8_15_Record"] == True)
    &
    (
        (check["Oil_BPD"].fillna(0) < 10)
        |
        (check["Gas_MCF"].fillna(0) <= 0)
    )
    &
    (~zero_mask)
)
# ============================================================
# 6. ASSIGN ISSUE CATEGORY
# ============================================================
check["Issue_Category"] = ""
check.loc[
    zero_mask,
    "Issue_Category"
] = "A - ZERO"
check.loc[
    missing_mask,
    "Issue_Category"
] = "B - MISSING"
check.loc[
    low_oil_or_no_gas_mask,
    "Issue_Category"
] = "C - LOW OIL / NO GAS"
# ============================================================
# 7. KEEP ONLY WELLS WITH AN ISSUE
# ============================================================
issues = check[
    check["Issue_Category"] != ""
].copy()
# Keep missing production blank.
# Do NOT turn missing production into zero.
issues.loc[
    issues["Issue_Category"] == "B - MISSING",
    [
        "Oil_BBL",
        "Oil_BPD",
        "Gas_MCF",
        "Gas_MCFD"
    ]
] = pd.NA
# ============================================================
# 8. RENAME COLUMNS FOR OUTPUT
# ============================================================
issues = issues.rename(
    columns={
        "Match_Name":
            "Well_Name",
        "Oil_BBL":
            "Aug_15_Oil_BBL",
        "Oil_BPD":
            "Aug_15_Oil_BPD",
        "Gas_MCF":
            "Aug_15_Gas_MCF",
        "Gas_MCFD":
            "Aug_15_Gas_MCFD"
    }
)
# ============================================================
# 9. FINAL COLUMN ORDER
# ============================================================
issues = issues[
    [
        "Issue_Category",
        "Well_Name",
        "Routing_Category",
        "Facility",
        "WI",
        "Aug_15_Oil_BBL",
        "Aug_15_Oil_BPD",
        "Aug_15_Gas_MCF",
        "Aug_15_Gas_MCFD"
    ]
]
# ============================================================
# 10. SORT
# ============================================================
issues = (
    issues
    .sort_values(
        [
            "Issue_Category",
            "Routing_Category",
            "Facility",
            "Well_Name"
        ]
    )
    .reset_index(drop=True)
)
# ============================================================
# 11. CREATE SEPARATE CATEGORY TABLES
# ============================================================
zero_wells = issues[
    issues["Issue_Category"] == "A - ZERO"
].copy()
missing_wells = issues[
    issues["Issue_Category"] == "B - MISSING"
].copy()
low_oil_or_no_gas_wells = issues[
    issues["Issue_Category"] ==
    "C - LOW OIL / NO GAS"
].copy()
# ============================================================
# 12. SAVE FULL ISSUE LIST
# ============================================================
issues.to_csv(
    OUTPUT_FILE,
    index=False
)
# ============================================================
# 13. SUMMARY
# ============================================================
print("=" * 75)
print("8/15/2026 PRODUCTION ISSUE CHECK")
print("=" * 75)
print(
    "\nTotal unique routed wells:",
    route_summary["Match_Name"].nunique()
)
print(
    "Wells with an 8/15/2026 production record:",
    int(check["Has_8_15_Record"].sum())
)
print(
    "\nA - ZERO:",
    len(zero_wells)
)
print(
    "B - MISSING:",
    len(missing_wells)
)
print(
    "C - LOW OIL / NO GAS:",
    len(low_oil_or_no_gas_wells)
)
print(
    "\nTOTAL WELLS FLAGGED:",
    len(issues)
)
print(
    "\nSaved:",
    OUTPUT_FILE
)
# ============================================================
# 14. FULL OUTPUT
# ============================================================
print("\n" + "=" * 75)
print("ALL FLAGGED WELLS")
print("=" * 75)
display(issues)
# ============================================================
# 15. CATEGORY A
# ============================================================
print("\n" + "=" * 75)
print("A - ZERO PRODUCTION")
print("=" * 75)
display(zero_wells)
# ============================================================
# 16. CATEGORY B
# ============================================================
print("\n" + "=" * 75)
print("B - MISSING PRODUCTION")
print("=" * 75)
display(missing_wells)
# ============================================================
# 17. CATEGORY C
# ============================================================
print("\n" + "=" * 75)
print("C - LOW OIL (<10 BPD) OR NO GAS")
print("=" * 75)
display(low_oil_or_no_gas_wells)


In [ ]:
# ============================================================
# 8/15/2026 PRODUCTION ISSUE DIAGNOSTIC
#
# A - ZERO
#     Has 8/15/2026 record
#     AND Oil = 0 AND Gas = 0
#
# B - MISSING
#     Routed well has NO 8/15/2026 production record
#
# C - LOW OIL / LOW GAS
#     Has 8/15/2026 record
#     AND:
#         Oil < 10 BPD
#         OR
#         Gas < 10 MCF/D
#
# True zero wells are classified as A, not C.
#
# Uses objects already created in prior cell:
#     routing_all
#     prod_815
#
# Output:
#     08_15_2026_Production_Issues.csv
# ============================================================
import pandas as pd
OUTPUT_FILE = "08_15_2026_Production_Issues.csv"
DAYS_IN_AUGUST = 31
# ============================================================
# 1. CREATE ONE ROW PER ROUTED WELL
# ============================================================
routes_check = routing_all.copy()
# Preserve multiple category/facility labels if a duplicate
# routing exists (for example, the Teton duplicate).
route_summary = (
    routes_check
    .groupby(
        "Match_Name",
        as_index=False
    )
    .agg(
        Facility=(
            "Facility",
            lambda x: " | ".join(
                sorted(set(x.dropna().astype(str)))
            )
        ),
        Routing_Category=(
            "Category",
            lambda x: " | ".join(
                sorted(set(x.dropna().astype(str)))
            )
        ),
        WI=(
            "WI",
            "first"
        )
    )
)
# ============================================================
# 2. CREATE ONE 8/15/2026 PRODUCTION ROW PER WELL
# ============================================================
prod_check = (
    prod_815
    .groupby(
        "Match_Name",
        as_index=False
    )[
        [
            "Oil_BBL",
            "Gas_MCF"
        ]
    ]
    .sum()
)
# Explicit flag so MISSING is not confused with ZERO
prod_check["Has_8_15_Record"] = True
# ============================================================
# 3. MERGE ROUTED WELL POPULATION TO PRODUCTION
# ============================================================
check = route_summary.merge(
    prod_check,
    on="Match_Name",
    how="left"
)
check["Has_8_15_Record"] = (
    check["Has_8_15_Record"]
    .fillna(False)
    .astype(bool)
)
# ============================================================
# 4. CALCULATE DAILY RATES
# ============================================================
check["Oil_BPD"] = (
    check["Oil_BBL"]
    / DAYS_IN_AUGUST
)
check["Gas_MCFD"] = (
    check["Gas_MCF"]
    / DAYS_IN_AUGUST
)
# ============================================================
# 5. DEFINE ISSUE CATEGORIES
# ============================================================
# ------------------------------------------------------------
# B - MISSING
#
# Well exists in routing but there is no 8/15/2026
# production record in the source file.
# ------------------------------------------------------------
missing_mask = (
    check["Has_8_15_Record"] == False
)
# ------------------------------------------------------------
# A - ZERO
#
# Production record exists but BOTH oil and gas are zero.
# ------------------------------------------------------------
zero_mask = (
    (check["Has_8_15_Record"] == True)
    &
    (check["Oil_BBL"].fillna(0) == 0)
    &
    (check["Gas_MCF"].fillna(0) == 0)
)
# ------------------------------------------------------------
# C - LOW OIL / LOW GAS
#
# OR LOGIC:
#
#     Oil < 10 BPD
#              OR
#     Gas < 10 MCF/D
#
# Excludes true zero wells because those belong in A.
# ------------------------------------------------------------
low_oil_or_low_gas_mask = (
    (check["Has_8_15_Record"] == True)
    &
    (
        (check["Oil_BPD"].fillna(0) < 10)
        |
        (check["Gas_MCFD"].fillna(0) < 10)
    )
    &
    (~zero_mask)
)
# ============================================================
# 6. ASSIGN ISSUE CATEGORY
# ============================================================
check["Issue_Category"] = ""
check.loc[
    zero_mask,
    "Issue_Category"
] = "A - ZERO"
check.loc[
    missing_mask,
    "Issue_Category"
] = "B - MISSING"
check.loc[
    low_oil_or_low_gas_mask,
    "Issue_Category"
] = "C - LOW OIL / LOW GAS"
# ============================================================
# 7. KEEP ONLY WELLS WITH AN ISSUE
# ============================================================
issues = check[
    check["Issue_Category"] != ""
].copy()
# Leave missing production blank.
# Do NOT convert missing values to zero.
issues.loc[
    issues["Issue_Category"] == "B - MISSING",
    [
        "Oil_BBL",
        "Oil_BPD",
        "Gas_MCF",
        "Gas_MCFD"
    ]
] = pd.NA
# ============================================================
# 8. RENAME COLUMNS FOR OUTPUT
# ============================================================
issues = issues.rename(
    columns={
        "Match_Name":
            "Well_Name",
        "Oil_BBL":
            "Aug_15_Oil_BBL",
        "Oil_BPD":
            "Aug_15_Oil_BPD",
        "Gas_MCF":
            "Aug_15_Gas_MCF",
        "Gas_MCFD":
            "Aug_15_Gas_MCFD"
    }
)
# ============================================================
# 9. FINAL COLUMN ORDER
# ============================================================
issues = issues[
    [
        "Issue_Category",
        "Well_Name",
        "Routing_Category",
        "Facility",
        "WI",
        "Aug_15_Oil_BBL",
        "Aug_15_Oil_BPD",
        "Aug_15_Gas_MCF",
        "Aug_15_Gas_MCFD"
    ]
]
# ============================================================
# 10. SORT
# ============================================================
issues = (
    issues
    .sort_values(
        [
            "Issue_Category",
            "Routing_Category",
            "Facility",
            "Well_Name"
        ]
    )
    .reset_index(drop=True)
)
# ============================================================
# 11. CREATE SEPARATE CATEGORY TABLES
# ============================================================
zero_wells = issues[
    issues["Issue_Category"] == "A - ZERO"
].copy()
missing_wells = issues[
    issues["Issue_Category"] == "B - MISSING"
].copy()
low_oil_or_low_gas_wells = issues[
    issues["Issue_Category"] ==
    "C - LOW OIL / LOW GAS"
].copy()
# ============================================================
# 12. SAVE FULL ISSUE LIST
# ============================================================
issues.to_csv(
    OUTPUT_FILE,
    index=False
)
# Also save separate category files
zero_wells.to_csv(
    "08_15_2026_A_ZERO_Wells.csv",
    index=False
)
missing_wells.to_csv(
    "08_15_2026_B_MISSING_Wells.csv",
    index=False
)
low_oil_or_low_gas_wells.to_csv(
    "08_15_2026_C_LOW_OIL_OR_LOW_GAS_Wells.csv",
    index=False
)
# ============================================================
# 13. SUMMARY
# ============================================================
print("=" * 75)
print("8/15/2026 PRODUCTION ISSUE CHECK")
print("=" * 75)
print(
    "\nTotal unique routed wells:",
    route_summary["Match_Name"].nunique()
)
print(
    "Wells with an 8/15/2026 production record:",
    int(check["Has_8_15_Record"].sum())
)
print(
    "\nA - ZERO:",
    len(zero_wells)
)
print(
    "B - MISSING:",
    len(missing_wells)
)
print(
    "C - LOW OIL / LOW GAS:",
    len(low_oil_or_low_gas_wells)
)
print(
    "\nTOTAL WELLS FLAGGED:",
    len(issues)
)
print(
    "\nSaved:",
    OUTPUT_FILE
)
print(
    "Saved:",
    "08_15_2026_A_ZERO_Wells.csv"
)
print(
    "Saved:",
    "08_15_2026_B_MISSING_Wells.csv"
)
print(
    "Saved:",
    "08_15_2026_C_LOW_OIL_OR_LOW_GAS_Wells.csv"
)
# ============================================================
# 14. FULL OUTPUT
# ============================================================
print("\n" + "=" * 75)
print("ALL FLAGGED WELLS")
print("=" * 75)
display(issues)
# ============================================================
# 15. CATEGORY A
# ============================================================
print("\n" + "=" * 75)
print("A - ZERO PRODUCTION")
print("=" * 75)
display(zero_wells)
# ============================================================
# 16. CATEGORY B
# ============================================================
print("\n" + "=" * 75)
print("B - MISSING PRODUCTION")
print("=" * 75)
display(missing_wells)
# ============================================================
# 17. CATEGORY C
# ============================================================
print("\n" + "=" * 75)
print("C - LOW OIL (<10 BPD) OR LOW GAS (<10 MCF/D)")
print("=" * 75)
display(low_oil_or_low_gas_wells)


In [ ]:
# Hercules export treatment is now integrated upstream in the 01-05 export cell.
# This former late re-export patch is intentionally disabled to avoid duplicate/conflicting files.
